# Differentially-Private Synthetic Data for Uplift Modelling

**Student number:** S4721985

This notebook is the complete, reproducible pipeline for the thesis. Run from top to bottom, it
regenerates every table, figure, and CSV reported in the work: the main analysis — building
differentially-private synthetic versions of the Hillstrom (2008) e-mail dataset with the MST
mechanism and evaluating two-model uplift performance across the privacy budget ε — together with
two extensions (a pooling-robustness study and an analysis of the ε "sweet spot").

All analysis code and random seeds are fixed (master seed **42**), so a clean run is fully
deterministic and reproduces the reported results.

## Self-contained by design
The notebook reads a single input — the bundled Hillstrom CSV — and writes every output inside one
project folder, `PROJECT_ROOT` (this repository). Nothing is written outside it.

## How to run
1. **Run from this folder** so `PROJECT_ROOT` is this folder automatically (or set the
   `PROJECT_ROOT` environment variable to a chosen output folder).
2. Set up the environment: `conda env create -f environment.yml` (or
   `pip install -r requirements.txt`).
3. The MST library (`private-pgm`) is bundled in this repository and is located automatically.
4. The Hillstrom CSV is already in **`data/raw/`** — the only required input.
5. **Run all cells, top to bottom.** §0 reports the status of each prerequisite before any heavy
   computation begins.

## Project layout
```
<repo>/  (= PROJECT_ROOT)
├── data/
│   ├── raw/                  # input: the Hillstrom CSV (included)
│   ├── main_benchmarks/      # train/test split and benchmark datasets
│   ├── main_synthetic_mst/   # MST differentially-private synthetic datasets
│   └── extra_analysis/       # datasets for the two extensions
└── results/
    ├── main/{raw_outputs,thesis_tables,figures}/
    └── extra_analysis/{epsilon_elbow,pooling_robustness,sweetspot_v2}/
```

## Contents
| Section | Reads | Writes |
|---|---|---|
| **§0 — Setup** | — | creates the project folder tree |
| **§1 — Main pipeline** | `data/raw/` | `data/main_benchmarks/`, `data/main_synthetic_mst/`, `data/extra_analysis/epsilon_elbow/`, `results/main/{raw_outputs,thesis_tables,figures}/`, `results/extra_analysis/epsilon_elbow/` |
| **§2 — Extension: pooling robustness** | `data/raw/` | `data/extra_analysis/pooling_robustness/`, `results/extra_analysis/pooling_robustness/raw_outputs/` |
| **§3 — Extension: the ε sweet spot** | `data/main_benchmarks/`, `results/main/raw_outputs/results_summary.csv` | `data/extra_analysis/sweetspot_v2/`, `results/extra_analysis/sweetspot_v2/{raw_outputs,figures}/` |


In [1]:
# =============================================================================
# §0 — Configuration  (the single source of truth for every path)
# =============================================================================
# The pipeline reads the raw Hillstrom CSV and reads/writes ONLY inside PROJECT_ROOT.
import os
from pathlib import Path

# OUTPUT root = this repository. Everything the pipeline writes goes here, nowhere else.
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).resolve()

# private-pgm mechanisms folder (MST lives here). Prefer the copy bundled inside this repo,
# then a clone beside the repo; either can be overridden with the PRIVATE_PGM_MECHANISMS env var.
def _find_mechanisms():
    env = os.environ.get("PRIVATE_PGM_MECHANISMS")
    if env:
        return Path(env)
    for cand in (PROJECT_ROOT / "private-pgm" / "mechanisms",
                 PROJECT_ROOT.parent / "private-pgm" / "mechanisms"):
        if cand.exists():
            return cand
    return PROJECT_ROOT / "private-pgm" / "mechanisms"

PRIVATE_PGM_MECHANISMS = _find_mechanisms().resolve()

# -----------------------------------------------------------------------------
# Output tree. Created on run; deeper sub-folders are made by the pipeline cells.
# -----------------------------------------------------------------------------
OUTPUT_TREE = [
    "data/raw",
    "data/main_benchmarks",
    "data/main_synthetic_mst",
    "data/extra_analysis",
    "results/main/raw_outputs",
    "results/main/thesis_tables",
    "results/main/figures",
    "results/extra_analysis/epsilon_elbow",
    "results/extra_analysis/pooling_robustness",
    "results/extra_analysis/sweetspot_v2",
]
for _rel in OUTPUT_TREE:
    (PROJECT_ROOT / _rel).mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# Status report
# -----------------------------------------------------------------------------
_RAW_CSV_NAME = "Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv"
_raw_csv = PROJECT_ROOT / "data" / "raw" / _RAW_CSV_NAME


def _status(label, path, is_file=False, note=""):
    ok = path.is_file() if is_file else path.exists()
    print(f"  {'[ok]  ' if ok else '[warn]'} {label:<26} {path}{note}")
    return ok


print("=" * 78)
print("§0 CONFIG — output tree rooted at PROJECT_ROOT (nothing is written elsewhere)")
print("=" * 78)
_status("PROJECT_ROOT", PROJECT_ROOT)
_status("PRIVATE_PGM_MECHANISMS", PRIVATE_PGM_MECHANISMS, note="  (need it before the MST cells)")
_status("raw CSV (data/raw/)", _raw_csv, is_file=True, note="  (only required input)")

print("\nDirectory map (created with mkdir(parents=True, exist_ok=True)):")
print(f"  {PROJECT_ROOT}")
print(r"""  ├── data/
  │   ├── raw/                  # INPUT: the Hillstrom CSV (only required input)
  │   ├── main_benchmarks/
  │   ├── main_synthetic_mst/
  │   └── extra_analysis/
  └── results/
      ├── main/{raw_outputs,thesis_tables,figures}/
      └── extra_analysis/{epsilon_elbow,pooling_robustness,sweetspot_v2}/""")
print("\n[ok] §0 complete — paths configured, output tree present.")


§0 CONFIG — output tree rooted at PROJECT_ROOT (nothing is written elsewhere)
  [ok]   PROJECT_ROOT               /Users/tumtummer/Downloads/dp-uplift-modelling
  [ok]   PRIVATE_PGM_MECHANISMS     /Users/tumtummer/Downloads/dp-uplift-modelling/private-pgm/mechanisms  (need it before the MST cells)
  [ok]   raw CSV (data/raw/)        /Users/tumtummer/Downloads/dp-uplift-modelling/data/raw/Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv  (only required input)

Directory map (created with mkdir(parents=True, exist_ok=True)):
  /Users/tumtummer/Downloads/dp-uplift-modelling
  ├── data/
  │   ├── raw/                  # INPUT: the Hillstrom CSV (only required input)
  │   ├── main_benchmarks/
  │   ├── main_synthetic_mst/
  │   └── extra_analysis/
  └── results/
      ├── main/{raw_outputs,thesis_tables,figures}/
      └── extra_analysis/{epsilon_elbow,pooling_robustness,sweetspot_v2}/

[ok] §0 complete — paths configured, output tree present.


# §1 — Main pipeline

The Hillstrom data are prepared and validated, and the benchmark datasets (size variants and
subgroup-imbalance variants) are constructed. Differentially-private synthetic data are generated
with the MST mechanism, two-model uplift is estimated, and ranking quality (Qini) is evaluated
across the privacy budget ε — together with the ε-elbow analysis, the negative-ρ response-surface
diagnostic, and all Chapter-4 tables and figures.


In [2]:
# =============================================================================
# Project paths — clean output structure
# =============================================================================

from pathlib import Path
import os


DATA_ROOT = PROJECT_ROOT / "data"
RESULTS_ROOT = PROJECT_ROOT / "results"

RAW_DATA_DIR = DATA_ROOT / "raw"
DATA_DIR = DATA_ROOT / "main_benchmarks"
SYNTH_DIR = DATA_ROOT / "main_synthetic_mst"

RESULTS_DIR = RESULTS_ROOT / "main" / "raw_outputs"
TABLES_DIR = RESULTS_ROOT / "main" / "thesis_tables"
FIGURES_DIR = RESULTS_ROOT / "main" / "figures"

EXTRA_ROOT = RESULTS_ROOT / "extra_analysis"
ELBOW_RESULTS_DIR = EXTRA_ROOT / "epsilon_elbow" / "raw_outputs"
ELBOW_TABLES_DIR = EXTRA_ROOT / "epsilon_elbow" / "thesis_tables"
ELBOW_FIGURES_DIR = EXTRA_ROOT / "epsilon_elbow" / "figures"
ELBOW_SYNTH_DIR = DATA_ROOT / "extra_analysis" / "epsilon_elbow" / "synthetic_mst"

for p in [
    RAW_DATA_DIR, DATA_DIR, SYNTH_DIR,
    RESULTS_DIR, TABLES_DIR, FIGURES_DIR,
    ELBOW_RESULTS_DIR, ELBOW_TABLES_DIR, ELBOW_FIGURES_DIR, ELBOW_SYNTH_DIR
]:
    p.mkdir(parents=True, exist_ok=True)

# Backwards compatibility for existing code that expects strings.
DATA_DIR = str(DATA_DIR) + os.sep
SYNTH_DIR = str(SYNTH_DIR) + os.sep
RESULTS_DIR = str(RESULTS_DIR) + os.sep
TABLES_DIR = str(TABLES_DIR) + os.sep
FIGURES_DIR = str(FIGURES_DIR) + os.sep
ELBOW_RESULTS_DIR = str(ELBOW_RESULTS_DIR) + os.sep
ELBOW_TABLES_DIR = str(ELBOW_TABLES_DIR) + os.sep
ELBOW_FIGURES_DIR = str(ELBOW_FIGURES_DIR) + os.sep
ELBOW_SYNTH_DIR = str(ELBOW_SYNTH_DIR) + os.sep

In [3]:
"""
Phase 1 — Data Foundation
=========================
Steps 1–3 of the analysis procedure (Section 3.1):
  1. Data preparation: load, validate, recode, and split the Hillstrom dataset
  2. Descriptive baseline comparison: treatment vs control means (Section 3.3)
  3. Benchmark dataset construction: size variants + subgroup-imbalance variants (Section 3.1.4)

Output: train.csv, test.csv, and seven benchmark CSVs saved to OUT_DIR
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os
import warnings
warnings.filterwarnings('ignore')


# =============================================================================
# CONFIGURATION
# =============================================================================
# Paths
RAW_PATH = os.path.join(RAW_DATA_DIR, "Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv")
OUT_DIR = DATA_DIR
os.makedirs(OUT_DIR, exist_ok=True)

# Random seed: fixed at 42 throughout so every run produces identical splits
RANDOM_SEED = 42

# Column names used throughout the pipeline
FEATURES       = ['recency', 'history_segment', 'mens', 'womens', 'zip_code', 'newbie', 'channel']
OUTCOME_BINARY = 'conversion'   # primary binary outcome (0/1)
OUTCOME_SPEND  = 'spend'        # continuous spend outcome
OUTCOME_VISIT  = 'visit'        # secondary binary outcome (0/1)
TREATMENT      = 'treatment'    # recoded binary treatment indicator


# =============================================================================
# STEP 1: DATA PREPARATION
# =============================================================================
print("=" * 70)
print("STEP 1: DATA PREPARATION")
print("=" * 70)

# --- 1a. Load raw Hillstrom (2008) dataset ---
df = pd.read_csv(RAW_PATH)
print(f"\nRaw dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")


# --- 1b. Preprocessing checks (Section 3.1.3) ---
# These assertions verify the dataset matches the published description exactly.
# Any failure here would indicate a file or merge error.
print("\n--- Preprocessing Checks ---")

assert df.isnull().sum().sum() == 0, "Missing values found!"
print("[OK] No missing values")

assert df.shape[0] == 64000, f"Expected 64,000 rows, got {df.shape[0]}"
print("[OK] 64,000 observations confirmed")

seg_counts = df['segment'].value_counts()
print(f"[OK] Treatment segments: {dict(seg_counts)}")

# Binary outcomes must be strictly 0/1 — a requirement of the Two-Model uplift
assert df['visit'].isin([0, 1]).all(), "visit is not binary!"
assert df['conversion'].isin([0, 1]).all(), "conversion is not binary!"
print("[OK] visit and conversion are binary")

# Spend must be non-negative and zero for every non-converter (data integrity)
assert (df['spend'] >= 0).all(), "Negative spend found!"
assert (df.loc[df['conversion'] == 0, 'spend'] == 0).all(), "Non-zero spend for non-converters!"
print("[OK] spend is non-negative and 0 for non-converters")

print("[OK] Outcomes measured during 2-week post-campaign window (per Hillstrom, 2008)")

n_dupes = df.duplicated().sum()
print(f"[OK] Duplicate rows: {n_dupes} (categorical features — identical profiles expected)")

print(f"[OK] recency range: {df['recency'].min()}-{df['recency'].max()} ({df['recency'].nunique()} unique)")

# thesis Table A1 - Spend distribution and discretisation
# --- Spend distribution check (Section 3.1.2 outlier rationale) ---
# Documents the right-skew that motivates the bin discretisation in Section 3.2.1.
print("\n--- Spend distribution check ---")
print(df['spend'].describe())
print(f"Pct with spend = 0: {(df['spend'] == 0).mean() * 100:.2f}%")
print(f"Observations with spend > 200: {(df['spend'] > 200).sum()}")

# --- 1c. Treatment recoding: binary (email vs no-email) ---
# Section 3.1.3: the original three segments (Men's, Women's, No E-Mail) are
# collapsed into a single binary treatment indicator: 1 = received any email.
# The original segment label is preserved for robustness checks later.
df['treatment'] = (df['segment'] != 'No E-Mail').astype(int)
df['segment_original'] = df['segment']

print(f"\n--- Treatment Recoding ---")
print(f"Treatment (any email): {df['treatment'].sum():,} ({df['treatment'].mean():.1%})")
print(f"Control (no email):    {(1 - df['treatment']).sum():,} ({(1 - df['treatment']).mean():.1%})")


# --- 1d. Feature set summary (Section 3.2.2) ---
# All seven pre-treatment features are already discrete (no scaling needed).
# MST synthesises each column as a categorical variable, so no transformation
# is applied here; recency is integer (1–12), the rest are nominal or binary.
print(f"\n--- Feature Set (all discrete, MST-ready) ---")
for feat in FEATURES:
    n_unique = df[feat].nunique()
    if n_unique <= 3:
        vals = sorted(df[feat].unique())
        print(f"  {feat}: {n_unique} levels — {vals}")
    else:
        print(f"  {feat}: {n_unique} levels (range: {df[feat].min()}-{df[feat].max()})")


# --- 1e. Stratified 70/30 train-test split (Section 3.1.3) ---
# Stratification key combines treatment AND conversion so both marginal
# distributions are preserved in both halves. The test set is held out
# entirely and only used in Phase 3 for evaluation.
df['_strat'] = df[TREATMENT].astype(str) + '_' + df[OUTCOME_BINARY].astype(str)

df_train, df_test = train_test_split(
    df, test_size=0.30, random_state=RANDOM_SEED, stratify=df['_strat']
)
df_train = df_train.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print(f"\n--- Train-Test Split (70/30, stratified on treatment × conversion) ---")
print(f"Training set: {df_train.shape[0]:,} rows")
print(f"Test set:     {df_test.shape[0]:,} rows")
print(f"Treatment ratio — train: {df_train[TREATMENT].mean():.3f}, test: {df_test[TREATMENT].mean():.3f}")
print(f"Conversion rate — train: {df_train[OUTCOME_BINARY].mean():.4f}, test: {df_test[OUTCOME_BINARY].mean():.4f}")

n_conv_treated_train = df_train.loc[df_train[TREATMENT] == 1, OUTCOME_BINARY].sum()
n_conv_control_train = df_train.loc[df_train[TREATMENT] == 0, OUTCOME_BINARY].sum()
print(f"\nPositive conversions (training): treatment={n_conv_treated_train}, control={n_conv_control_train}")

# Store the baseline newbie share; used later to define the imbalance_baseline variant
newbie_share_train = df_train['newbie'].mean()
print(f"Baseline newbie share (training): {newbie_share_train:.1%}")


# =============================================================================
# thesis Table 2 - Baseline treatment-control comparison (full dataset and training set)
# STEP 2: DESCRIPTIVE BASELINE COMPARISON
# =============================================================================
print("\n" + "=" * 70)
print("STEP 2: DESCRIPTIVE BASELINE TABLE")
print("=" * 70)

def baseline_table(data, label=""):
    """
    Compute treated-vs-control means for visit, conversion, and spend.
    The difference (T – C) is the naive ATE estimate before any modelling.
    """
    treated = data[data[TREATMENT] == 1]
    control = data[data[TREATMENT] == 0]
    rows = []
    for outcome in ['visit', 'conversion', 'spend']:
        t_mean = treated[outcome].mean()
        c_mean = control[outcome].mean()
        rows.append({
            'Outcome':      outcome,
            'Treated mean': round(t_mean, 5),
            'Control mean': round(c_mean, 5),
            'Uplift (T-C)': round(t_mean - c_mean, 5)
        })
    result = pd.DataFrame(rows).set_index('Outcome')
    if label:
        print(f"\n{label}:")
    print(result.to_string())
    return result

# Baseline on the full dataset — cross-check against Radcliffe (2008)
baseline_full  = baseline_table(df, f"Baseline (full dataset, N={df.shape[0]:,})")

# Baseline on training set — reference point for all subsequent modelling steps
baseline_train = baseline_table(df_train, f"Baseline (training set, N={df_train.shape[0]:,})")

# thesis Table 3 - Per-segment descriptive comparison by treatment condition
# Per-segment breakdown (Men's vs Women's) — provides robustness context
print("\n--- Per-segment breakdown ---")
ctrl_data = df_train[df_train['segment_original'] == 'No E-Mail']
for seg in ['Mens E-Mail', 'Womens E-Mail']:
    seg_data = df_train[df_train['segment_original'] == seg]
    print(f"\n  {seg} vs Control:")
    for outcome in ['visit', 'conversion', 'spend']:
        t_mean = seg_data[outcome].mean()
        c_mean = ctrl_data[outcome].mean()
        print(f"    {outcome}: {t_mean:.4f} vs {c_mean:.4f} (uplift: {t_mean - c_mean:+.4f})")


# =============================================================================
# STEP 3: CONSTRUCT BENCHMARK DATASET VARIANTS
# =============================================================================
print("\n" + "=" * 70)
print("STEP 3: BENCHMARK DATASET VARIANTS")
print("=" * 70)

# thesis Table 4 - Dataset-size training variants (construction validated in Table A4)
# --- 3a. Dataset-size variants (Section 3.1.4) ---
# Four stratified subsamples of the training set: 25%, 50%, 75%, 100%.
# These test whether DP utility degrades faster when fewer training rows exist.
SIZE_FRACTIONS = [0.25, 0.50, 0.75, 1.00]

size_variants = {}
print("\n--- Dataset-size variants ---")
print(f"{'Variant':<12} {'N':>7} {'Treat. ratio':>13} {'Conv (T)':>9} {'Newbie %':>9}")
print("-" * 55)

for frac in SIZE_FRACTIONS:
    if frac == 1.0:
        # Full training set — no subsampling needed
        variant = df_train.copy()
    else:
        # Stratified subsample (same stratification key as the global split)
        variant, _ = train_test_split(
            df_train, train_size=frac, random_state=RANDOM_SEED,
            stratify=df_train['_strat']
        )
        variant = variant.reset_index(drop=True)

    key = f"size_{int(frac * 100)}"
    size_variants[key] = variant

    n_conv = variant.loc[variant[TREATMENT] == 1, OUTCOME_BINARY].sum()
    print(f"{key:<12} {variant.shape[0]:>7,} {variant[TREATMENT].mean():>13.3f} "
          f"{n_conv:>9} {variant['newbie'].mean():>8.1%}")


# thesis Table 5 - Subgroup-imbalance training variants
# --- 3b. Subgroup-imbalance variants (Section 3.1.4) ---
# These variants keep N constant but resample to change the newbie share.
# They test whether MST struggles to synthesise rare subgroups under DP.
def create_imbalance_variant(data, target_newbie_share, random_state=42):
    """
    Resample df to achieve a target fraction of newbie=1 rows,
    holding total N and treatment ratio constant.
    Sampling with replacement is used when the target exceeds the available pool.
    """
    rng = np.random.RandomState(random_state)

    newbies  = data[data['newbie'] == 1].copy()
    existing = data[data['newbie'] == 0].copy()

    n_total           = len(data)
    n_target_newbies  = int(n_total * target_newbie_share)
    n_target_existing = n_total - n_target_newbies

    sampled_newbies  = newbies.sample(
        n=n_target_newbies,
        replace=(n_target_newbies > len(newbies)),
        random_state=rng
    )
    sampled_existing = existing.sample(
        n=n_target_existing,
        replace=(n_target_existing > len(existing)),
        random_state=rng
    )

    # Concatenate and shuffle so newbies are not all at the front
    result = pd.concat([sampled_newbies, sampled_existing], ignore_index=True)
    return result.sample(frac=1, random_state=rng).reset_index(drop=True)


# Three imbalance levels: natural baseline, 20% newbie, and 5% newbie (severe)
IMBALANCE_LEVELS = {
    'imbalance_baseline': newbie_share_train,  # natural share (~50%)
    'imbalance_20':       0.20,                # moderate minority
    'imbalance_05':       0.05,                # severe minority
}

imbalance_variants = {}
print("\n--- Subgroup-imbalance variants ---")
print(f"{'Variant':<22} {'N':>7} {'Newbie actual':>13} {'Target':>8} {'Treat. ratio':>13}")
print("-" * 68)

for key, target_share in IMBALANCE_LEVELS.items():
    if key == 'imbalance_baseline':
        # No resampling needed — training set already at natural share
        variant = df_train.copy()
    else:
        variant = create_imbalance_variant(df_train, target_share, random_state=RANDOM_SEED)

    imbalance_variants[key] = variant
    actual = variant['newbie'].mean()
    print(f"{key:<22} {variant.shape[0]:>7,} {actual:>12.1%} {target_share:>7.0%} "
          f"{variant[TREATMENT].mean():>13.3f}")


# --- 3c. Sanity check: test set must be untouched ---
print("\n--- Verification (test set unchanged) ---")
print(f"Test set: {df_test.shape[0]:,} rows | "
      f"treatment ratio: {df_test[TREATMENT].mean():.3f} | "
      f"conversion rate: {df_test[OUTCOME_BINARY].mean():.4f}")


# =============================================================================
# SAVE ALL DATA TO DISK
# =============================================================================
# Every file is written with index=False so no extra index column appears
# when the CSVs are read back in Phase 2 and Phase 3.
print("\n" + "=" * 70)
print("SAVING DATA")
print("=" * 70)

df_train.to_csv(f"{OUT_DIR}train.csv", index=False)
df_test.to_csv(f"{OUT_DIR}test.csv", index=False)
print(f"Saved: train.csv ({df_train.shape[0]:,} rows)")
print(f"Saved: test.csv ({df_test.shape[0]:,} rows)")

for key, variant in size_variants.items():
    variant.to_csv(f"{OUT_DIR}{key}.csv", index=False)
    print(f"Saved: {key}.csv ({variant.shape[0]:,} rows)")

for key, variant in imbalance_variants.items():
    variant.to_csv(f"{OUT_DIR}{key}.csv", index=False)
    print(f"Saved: {key}.csv ({variant.shape[0]:,} rows)")

print(f"\nAll files saved to: {OUT_DIR}")
print("\n*** Phase 1 complete. Ready for Phase 2 (MST synthesis). ***")


STEP 1: DATA PREPARATION

Raw dataset: 64,000 rows, 12 columns

--- Preprocessing Checks ---
[OK] No missing values
[OK] 64,000 observations confirmed
[OK] Treatment segments: {'Womens E-Mail': np.int64(21387), 'Mens E-Mail': np.int64(21307), 'No E-Mail': np.int64(21306)}
[OK] visit and conversion are binary
[OK] spend is non-negative and 0 for non-converters
[OK] Outcomes measured during 2-week post-campaign window (per Hillstrom, 2008)
[OK] Duplicate rows: 6562 (categorical features — identical profiles expected)
[OK] recency range: 1-12 (12 unique)

--- Spend distribution check ---
count    64000.000000
mean         1.050908
std         15.036448
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max        499.000000
Name: spend, dtype: float64
Pct with spend = 0: 99.10%
Observations with spend > 200: 93

--- Treatment Recoding ---
Treatment (any email): 42,694 (66.7%)
Control (no email):    21,306 (33.3%)

--- Feature Set (all discrete, MST-rea

# Randomization check

In [4]:
# ── Randomization Check: original groups + pooled treatment vs control ────────
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, ttest_ind

NUM = ['recency', 'history', 'mens', 'womens', 'newbie']   # Pre-treatment numeric/binary covariates
CAT = ['history_segment', 'zip_code', 'channel']           # Pre-treatment categorical covariates

PAIRS = [                                                  # All original assignment-group comparisons
    ('Mens E-Mail', 'No E-Mail'),
    ('Womens E-Mail', 'No E-Mail'),
    ('Mens E-Mail', 'Womens E-Mail')
]

needed = ['segment_original', 'treatment'] + NUM + CAT     # Columns required for this check
assert not set(needed) - set(df_train.columns), set(needed) - set(df_train.columns)


def smd(a, b):                                             # Standardized mean difference
    a, b = a.astype(float), b.astype(float)                # Ensure numeric comparison
    s = np.sqrt((a.var() + b.var()) / 2)                   # Pooled standard deviation
    return abs(a.mean() - b.mean()) / s if s else 0        # Mean difference scaled by SD


def flag(s):                                               # Simple SMD interpretation
    return '✗' if s >= .25 else '~' if s >= .10 else '✓'   # ✓ good, ~ watch, ✗ imbalance


def bal(data, group_col, g1, g2, num_vars=NUM, cat_vars=CAT): # Print balance for two groups
    d = data[data[group_col].isin([g1, g2])]               # Keep only compared groups
    a, b = d[d[group_col] == g1], d[d[group_col] == g2]    # Split into group 1 and group 2

    print(f"\n{g1} vs {g2}  (n={len(a):,} vs {len(b):,})") # Show sample sizes

    for f in num_vars:                                     # Numeric/binary covariates
        s = smd(a[f], b[f])                                # Compute SMD
        p = ttest_ind(a[f], b[f], equal_var=False).pvalue  # Welch t-test
        print(f"  {f:<18} SMD={s:.3f} {flag(s)}  p={p:.3f}")

    for f in cat_vars:                                     # Categorical covariates
        tab = pd.crosstab(d[group_col], d[f])              # Group x category table
        p = chi2_contingency(tab).pvalue                   # Chi-square test
        print(f"  {f:<18} chi-sq p={p:.3f}")


# 1. Check all original randomized groups pairwise
for g1, g2 in PAIRS:
    bal(df_train, 'segment_original', g1, g2)

# 2. Check pooled treatment: any email versus no email
df_pool = df_train.assign(
    pooled_treatment=df_train['treatment'].map({1: 'Email', 0: 'Control'}) # Pool Mens/Womens email
)
bal(df_pool, 'pooled_treatment', 'Email', 'Control')


# thesis Tables A2/A3 - Pre-treatment balance: original assignment groups; pooled e-mail vs control
# ── Minimal appendix tables: numeric SMDs + categorical chi-square p-values ──
def make_num_table(data, group_col, comparisons, num_vars): # Table for numeric/binary covariates
    rows = []
    for g1, g2 in comparisons:                              # Loop over requested comparisons
        d = data[data[group_col].isin([g1, g2])]             # Keep compared groups only
        a, b = d[d[group_col] == g1], d[d[group_col] == g2]  # Split groups
        for f in num_vars:                                  # One row per covariate
            rows.append({
                'comparison': f'{g1} vs {g2}',
                'covariate': f,
                g1: a[f].mean(),                            # Group 1 mean/proportion
                g2: b[f].mean(),                            # Group 2 mean/proportion
                'SMD': smd(a[f], b[f]),                     # Balance effect size
                'p_value': ttest_ind(a[f], b[f], equal_var=False).pvalue
            })
    return pd.DataFrame(rows).round(4)


def make_cat_table(data, group_col, comparisons, cat_vars): # Table for categorical covariates
    rows = []
    for g1, g2 in comparisons:                              # Loop over requested comparisons
        d = data[data[group_col].isin([g1, g2])]             # Keep compared groups only
        for f in cat_vars:                                  # One row per categorical variable
            tab = pd.crosstab(d[group_col], d[f])            # Group x category table
            rows.append({
                'comparison': f'{g1} vs {g2}',
                'covariate': f,
                'test': 'chi-square',
                'p_value': chi2_contingency(tab).pvalue
            })
    return pd.DataFrame(rows).round(4)


# Appendix A: original three assignment groups
appendix_num_original = make_num_table(df_train, 'segment_original', PAIRS, NUM)
appendix_cat_original = make_cat_table(df_train, 'segment_original', PAIRS, CAT)

display(appendix_num_original)                              # Numeric/binary balance table
display(appendix_cat_original)                              # Categorical balance p-value table


# Appendix B: pooled email treatment versus control
appendix_num_pooled = make_num_table(df_pool, 'pooled_treatment', [('Email', 'Control')], NUM)
appendix_cat_pooled = make_cat_table(df_pool, 'pooled_treatment', [('Email', 'Control')], CAT)

display(appendix_num_pooled)                                # Pooled numeric/binary balance table
display(appendix_cat_pooled)                                # Pooled categorical balance p-value table


# thesis Table A8 - Pre-treatment balance: subgroup-imbalance benchmark variants (newbie excluded)
# ── Subgroup-imbalance variants: pooled balance check excluding newbie ────────
VARIANT_NUM = [f for f in NUM if f != 'newbie']             # Exclude newbie because it is manipulated by design

variant_num_tables = []                                     # Store numeric balance tables per variant
variant_cat_tables = []                                     # Store categorical balance tables per variant

for name, variant in imbalance_variants.items():            # Check each imbalance dataset
    v_pool = variant.assign(
        pooled_treatment=variant['treatment'].map({1: 'Email', 0: 'Control'})
    )

    print(f"\nVariant balance check: {name} (newbie excluded)")
    bal(v_pool, 'pooled_treatment', 'Email', 'Control',
        num_vars=VARIANT_NUM, cat_vars=CAT)                 # Print diagnostic check

    num_tab = make_num_table(v_pool, 'pooled_treatment', [('Email', 'Control')], VARIANT_NUM)
    cat_tab = make_cat_table(v_pool, 'pooled_treatment', [('Email', 'Control')], CAT)

    num_tab.insert(0, 'variant', name)                      # Add variant name to table
    cat_tab.insert(0, 'variant', name)                      # Add variant name to table

    variant_num_tables.append(num_tab)
    variant_cat_tables.append(cat_tab)

appendix_num_variants = pd.concat(variant_num_tables, ignore_index=True)
appendix_cat_variants = pd.concat(variant_cat_tables, ignore_index=True)

display(appendix_num_variants)                              # Variant numeric/binary balance table
display(appendix_cat_variants)                              # Variant categorical balance p-value table

print("\nNote: p-values are unadjusted because these are diagnostic balance checks, not confirmatory tests.")

# ── Save balance check results ─────────────────────────────────────────────────
out_dir = RESULTS_ROOT / 'extra_analysis'
out_dir.mkdir(exist_ok=True)

pd.concat([appendix_num_original, appendix_num_pooled,
           appendix_cat_original, appendix_cat_pooled], ignore_index=True) \
  .to_csv(out_dir / 'balance_check_general.csv', index=False)

pd.concat([appendix_num_variants,
           appendix_cat_variants], ignore_index=True) \
  .to_csv(out_dir / 'balance_check_imbalance_variants.csv', index=False)


Mens E-Mail vs No E-Mail  (n=14,870 vs 14,914)
  recency            SMD=0.005 ✓  p=0.645
  history            SMD=0.019 ✓  p=0.107
  mens               SMD=0.001 ✓  p=0.923
  womens             SMD=0.007 ✓  p=0.560
  newbie             SMD=0.002 ✓  p=0.853
  history_segment    chi-sq p=0.182
  zip_code           chi-sq p=0.464
  channel            chi-sq p=0.723

Womens E-Mail vs No E-Mail  (n=15,016 vs 14,914)
  recency            SMD=0.015 ✓  p=0.181
  history            SMD=0.015 ✓  p=0.183
  mens               SMD=0.003 ✓  p=0.823
  womens             SMD=0.003 ✓  p=0.770
  newbie             SMD=0.000 ✓  p=1.000
  history_segment    chi-sq p=0.561
  zip_code           chi-sq p=0.986
  channel            chi-sq p=0.909

Mens E-Mail vs Womens E-Mail  (n=14,870 vs 15,016)
  recency            SMD=0.021 ✓  p=0.073
  history            SMD=0.004 ✓  p=0.761
  mens               SMD=0.004 ✓  p=0.748
  womens             SMD=0.003 ✓  p=0.771
  newbie             SMD=0.002 ✓  p=0.852
  hi

,comparison,covariate,Mens E-Mail,No E-Mail,SMD,p_value,Womens E-Mail
0,Mens E-Mail vs No E-Mail,recency,5.7253,5.7440,0.0053,0.6450,NaN
1,Mens E-Mail vs No E-Mail,history,243.4669,238.6644,0.0187,0.1073,NaN
2,Mens E-Mail vs No E-Mail,mens,0.5528,0.5522,0.0011,0.9229,NaN
3,Mens E-Mail vs No E-Mail,womens,0.5494,0.5460,0.0068,0.5598,NaN
4,Mens E-Mail vs No E-Mail,newbie,0.5012,0.5001,0.0022,0.8526,NaN
5,Womens E-Mail vs No E-Mail,recency,NaN,5.7440,0.0155,0.1812,5.7981
6,Womens E-Mail vs No E-Mail,history,NaN,238.6644,0.0154,0.1832,242.5564
7,Womens E-Mail vs No E-Mail,mens,NaN,0.5522,0.0026,0.8229,0.5509
8,Womens E-Mail vs No E-Mail,womens,NaN,0.5460,0.0034,0.7696,0.5477
9,Womens E-Mail vs No E-Mail,newbie,NaN,0.5001,0.0000,0.9999,0.5001


,comparison,covariate,test,p_value
0,Mens E-Mail vs No E-Mail,history_segment,chi-square,0.1823
1,Mens E-Mail vs No E-Mail,zip_code,chi-square,0.4636
2,Mens E-Mail vs No E-Mail,channel,chi-square,0.7225
3,Womens E-Mail vs No E-Mail,history_segment,chi-square,0.5612
4,Womens E-Mail vs No E-Mail,zip_code,chi-square,0.9858
5,Womens E-Mail vs No E-Mail,channel,chi-square,0.9091
6,Mens E-Mail vs Womens E-Mail,history_segment,chi-square,0.2975
7,Mens E-Mail vs Womens E-Mail,zip_code,chi-square,0.5133
8,Mens E-Mail vs Womens E-Mail,channel,chi-square,0.5582


,comparison,covariate,Email,Control,SMD,p_value
0,Email vs Control,recency,5.7619,5.7440,0.0051,0.6103
1,Email vs Control,history,243.0094,238.6644,0.0170,0.0877
2,Email vs Control,mens,0.5519,0.5522,0.0007,0.9410
3,Email vs Control,womens,0.5485,0.5460,0.0051,0.6135
4,Email vs Control,newbie,0.5007,0.5001,0.0011,0.9150


,comparison,covariate,test,p_value
0,Email vs Control,history_segment,chi-square,0.3459
1,Email vs Control,zip_code,chi-square,0.7416
2,Email vs Control,channel,chi-square,0.9177



Variant balance check: imbalance_baseline (newbie excluded)

Email vs Control  (n=29,886 vs 14,914)
  recency            SMD=0.005 ✓  p=0.610
  history            SMD=0.017 ✓  p=0.088
  mens               SMD=0.001 ✓  p=0.941
  womens             SMD=0.005 ✓  p=0.614
  history_segment    chi-sq p=0.346
  zip_code           chi-sq p=0.742
  channel            chi-sq p=0.918

Variant balance check: imbalance_20 (newbie excluded)

Email vs Control  (n=29,871 vs 14,929)
  recency            SMD=0.017 ✓  p=0.100
  history            SMD=0.004 ✓  p=0.667
  mens               SMD=0.017 ✓  p=0.091
  womens             SMD=0.012 ✓  p=0.246
  history_segment    chi-sq p=0.028
  zip_code           chi-sq p=0.345
  channel            chi-sq p=0.110

Variant balance check: imbalance_05 (newbie excluded)

Email vs Control  (n=29,882 vs 14,918)
  recency            SMD=0.010 ✓  p=0.312
  history            SMD=0.004 ✓  p=0.665
  mens               SMD=0.015 ✓  p=0.128
  womens             SMD=0.004 

,variant,comparison,covariate,Email,Control,SMD,p_value
0,imbalance_baseline,Email vs Control,recency,5.7619,5.7440,0.0051,0.6103
1,imbalance_baseline,Email vs Control,history,243.0094,238.6644,0.0170,0.0877
2,imbalance_baseline,Email vs Control,mens,0.5519,0.5522,0.0007,0.9410
3,imbalance_baseline,Email vs Control,womens,0.5485,0.5460,0.0051,0.6135
4,imbalance_20,Email vs Control,recency,5.8635,5.8057,0.0165,0.0996
5,imbalance_20,Email vs Control,history,208.3378,207.4981,0.0043,0.6667
6,imbalance_20,Email vs Control,mens,0.5485,0.5400,0.0169,0.0911
7,imbalance_20,Email vs Control,womens,0.5390,0.5448,0.0116,0.2459
8,imbalance_05,Email vs Control,recency,5.9137,5.8782,0.0101,0.3120
9,imbalance_05,Email vs Control,history,190.8574,190.2063,0.0043,0.6651


,variant,comparison,covariate,test,p_value
0,imbalance_baseline,Email vs Control,history_segment,chi-square,0.3459
1,imbalance_baseline,Email vs Control,zip_code,chi-square,0.7416
2,imbalance_baseline,Email vs Control,channel,chi-square,0.9177
3,imbalance_20,Email vs Control,history_segment,chi-square,0.0277
4,imbalance_20,Email vs Control,zip_code,chi-square,0.3447
5,imbalance_20,Email vs Control,channel,chi-square,0.1099
6,imbalance_05,Email vs Control,history_segment,chi-square,0.0296
7,imbalance_05,Email vs Control,zip_code,chi-square,0.0043
8,imbalance_05,Email vs Control,channel,chi-square,0.0483



Note: p-values are unadjusted because these are diagnostic balance checks, not confirmatory tests.


In [5]:
"""
Phase 2 — MST Synthesis
=======================
Step 4 of the analysis procedure (Section 3.2):
  For each benchmark variant (7 total), generate synthetic training sets using
  the Maximum Spanning Tree (MST) algorithm at seven privacy levels (ε),
  repeated 10 times per condition (Monte Carlo repetitions).

Input:  Seven benchmark CSVs from Phase 1 (DATA_DIR)
Output: 490 synthetic CSVs saved to SYNTH_DIR + synthesis_log.csv

Requires: private-pgm repo (github.com/ryan112358/private-pgm)
          Update MECHANISMS_PATH to point to its mechanisms/ folder.
"""

import sys
import os
import numpy as np
import pandas as pd
import time


# =============================================================================
# CONFIGURATION
# =============================================================================
# Path to the private-pgm mechanisms folder (contains mst.py and cdp2adp.py)
MECHANISMS_PATH = str(PRIVATE_PGM_MECHANISMS)
sys.path.insert(0, MECHANISMS_PATH)

os.makedirs(SYNTH_DIR, exist_ok=True)

# Privacy budget levels tested (Section 3.2.1).
# ε controls the privacy–utility trade-off: lower ε = stronger privacy = more noise.
EPSILON_LEVELS = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
DELTA          = 1e-5   # fixed δ for (ε, δ)-DP; small relative to N

# 10 Monte Carlo repetitions per condition: captures stochastic variance of MST
N_REPETITIONS = 10
BASE_SEED     = 42      # seed for rep r = BASE_SEED + r

# The seven benchmark variants from Phase 1
SIZE_VARIANTS      = ['size_25', 'size_50', 'size_75', 'size_100']
IMBALANCE_VARIANTS = ['imbalance_baseline', 'imbalance_20', 'imbalance_05']

# Columns passed to MST — spend is discretised into bins before synthesis
MST_COLS = ['recency', 'history_segment', 'mens', 'womens', 'zip_code',
            'newbie', 'channel', 'treatment', 'conversion', 'visit', 'spend_bin']

# Spend bin edges: 0, (0–50], (50–100], (100–200], (200–500]
# This converts a skewed continuous variable into 5 ordinal categories for MST.
SPEND_BINS = [-0.01, 0.01, 50, 100, 200, 500]

# Categorical columns that need integer encoding before MST
CAT_COLS = ['history_segment', 'zip_code', 'channel']


# =============================================================================
# LIBRARY IMPORTS (after sys.path is set)
# =============================================================================
from mbi import Dataset, Domain
from mst import MST as run_mst


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def prepare_for_mst(df):
    """
    Encode a benchmark DataFrame into the integer format required by MST.

    MST needs every column to be a non-negative integer starting at 0 because
    it builds a Domain object where each variable's cardinality = max value + 1.

    Steps:
      1. Label-encode categorical columns (history_segment, zip_code, channel)
         to 0-indexed integers; save the reverse mapping for decoding later.
      2. Discretise spend into 5 bins (SPEND_BINS) and store as spend_bin.
      3. Shift recency from 1–12 to 0–11 (MST requires 0-indexed domains).

    Returns: (encoded_df, encoding_maps) — encoding_maps is needed to reverse
    the transformation after synthesis (see decode_synthetic).
    """
    df = df.copy()
    encoding_maps = {}

    # Step 1: encode string categoricals as 0-indexed integers
    for col in CAT_COLS:
        cats = sorted(df[col].unique())          # sort ensures consistent ordering
        fwd  = {v: i for i, v in enumerate(cats)}   # string → integer
        bwd  = {i: v for v, i in fwd.items()}       # integer → string (for decoding)
        df[col] = df[col].map(fwd)
        encoding_maps[col] = bwd

    # Step 2: bin spend into 5 ordinal categories
    df['spend_bin'] = pd.cut(df['spend'], bins=SPEND_BINS, labels=range(5)).astype(int)

    # Step 3: shift recency to 0-indexed so the MST domain starts at 0
    df['recency'] = df['recency'] - 1

    return df, encoding_maps


def decode_synthetic(synth_df, encoding_maps, original_df):
    """
    Reverse the encoding applied in prepare_for_mst to restore the synthetic
    DataFrame to the same column format as the original benchmark CSVs.

    Steps:
      1. Reverse categorical encoding using the saved mapping dictionaries.
      2. Shift recency back from 0-indexed to 1–12.
      3. Map each spend_bin back to the median spend of that bin in the
         original data — this makes synthetic spend values realistic rather
         than arbitrary bin centres.
      4. Enforce spend = 0 for non-converters (mirrors the original data rule).
      5. Drop spend_bin and return only the columns present in Phase 1 CSVs.
    """
    df = synth_df.copy()

    # Step 1: reverse categorical encoding
    for col, bwd in encoding_maps.items():
        df[col] = df[col].map(bwd)

    # Step 2: undo the recency shift
    df['recency'] = df['recency'] + 1

    # Step 3: map spend_bin → representative spend value
    # Using the observed median per bin keeps synthetic spend distributions
    # anchored to the original data rather than to arbitrary bin midpoints.
    original = original_df.copy()
    original['spend_bin'] = pd.cut(original['spend'], bins=SPEND_BINS, labels=range(5)).astype(int)

    spend_map = {}
    for b in range(5):
        bin_data = original.loc[original['spend_bin'] == b, 'spend']
        if len(bin_data) > 0:
            spend_map[b] = bin_data.median()
        else:
            # Fallback to manual midpoints if a bin is empty (rare edge case)
            midpoints   = [0, 25, 75, 150, 350]
            spend_map[b] = midpoints[b]

    df['spend'] = df['spend_bin'].map(spend_map)

    # Step 4: enforce the data integrity rule: conversion=0 → spend=0
    df.loc[df['conversion'] == 0, 'spend'] = 0.0

    # Step 5: drop the helper column and return only original columns
    output_cols = ['recency', 'history_segment', 'mens', 'womens', 'zip_code',
                   'newbie', 'channel', 'treatment', 'conversion', 'visit', 'spend']
    return df[output_cols]


def generate_dp_synthetic(df, epsilon, delta, seed, is_no_dp=False):
    """
    Generate one synthetic dataset of the same size as df using MST at (ε, δ).

    MST procedure (McKenna et al., 2021):
      1. Select a set of two-way marginals using a private exponential mechanism.
      2. Measure those marginals with Gaussian noise calibrated to (ε, δ).
      3. Fit a graphical model (junction tree) to the noisy marginals.
      4. Sample synthetic records from the fitted model.

    The numpy random seed is set before calling MST so that multiple
    repetitions with different seeds produce independent synthetic datasets.
    """
    df_enc, encoding_maps = prepare_for_mst(df)

    # Build the Domain: a dictionary mapping column name → cardinality
    # Cardinality = max integer value + 1 (since values are 0-indexed)
    domain_dict = {col: int(df_enc[col].max()) + 1 for col in MST_COLS}
    domain      = Domain.fromdict(domain_dict)

    # Wrap encoded data in the MBI Dataset container
    data = Dataset(df_enc[MST_COLS].values.astype(int), domain)

    # Set the numpy seed and run MST
    np.random.seed(seed)
    synth_data = run_mst(data, epsilon=epsilon, delta=delta)

    # Convert the MBI Dataset back to a standard pandas DataFrame
    synth_dict = synth_data.to_dict()
    synth_df   = pd.DataFrame(synth_dict, columns=MST_COLS)

    # Decode back to the original variable format
    return decode_synthetic(synth_df, encoding_maps, df)


def generate_no_dp_synthetic(df, seed):
    """
    Generate synthetic data at ε = ∞ (no differential privacy).

    Using a very large ε (1e6) makes the Gaussian noise negligible, so MST
    effectively fits the true joint distribution and samples from it.
    This serves as the ε = ∞ baseline: the best MST can do without any
    privacy constraint.
    """
    # ε = 1e6 is a practical stand-in for ε → ∞ (noise ≈ 0)
    return generate_dp_synthetic(df, epsilon=1e6, delta=DELTA, seed=seed, is_no_dp=True)


# =============================================================================
# MAIN SYNTHESIS LOOP
# =============================================================================

if __name__ == "__main__":

    print("=" * 70)
    print("PHASE 2: MST SYNTHESIS")
    print("=" * 70)
    print(f"Epsilon levels: {EPSILON_LEVELS} + inf (no-DP baseline)")
    print(f"Delta: {DELTA}")
    print(f"Repetitions per condition: {N_REPETITIONS}")
    print(f"Benchmark variants: {SIZE_VARIANTS + IMBALANCE_VARIANTS}")

    all_variants = SIZE_VARIANTS + IMBALANCE_VARIANTS
    all_epsilons = EPSILON_LEVELS + ['inf']   # 'inf' triggers generate_no_dp_synthetic

    total_runs = len(all_variants) * len(all_epsilons) * N_REPETITIONS
    print(f"\nTotal synthesis runs: {total_runs}")   # 7 variants × 7 ε levels × 10 reps = 490

    results_log = []   # collects metadata for every run
    run_count   = 0
    start_time  = time.time()

    for variant_name in all_variants:

        # Load the benchmark variant CSV produced by Phase 1
        variant_path = os.path.join(DATA_DIR, f"{variant_name}.csv")
        df_variant   = pd.read_csv(variant_path)
        n_rows       = df_variant.shape[0]
        treat_ratio  = df_variant['treatment'].mean()
        conv_rate    = df_variant['conversion'].mean()

        print(f"\n{'=' * 50}")
        print(f"Variant: {variant_name} (N={n_rows:,}, treat={treat_ratio:.3f}, conv={conv_rate:.4f})")
        print(f"{'=' * 50}")

        for eps in all_epsilons:
            eps_label = f"eps_{eps}" if eps != 'inf' else "eps_inf"

            for rep in range(N_REPETITIONS):
                # Each repetition uses a unique seed (BASE_SEED + rep)
                seed = BASE_SEED + rep
                run_count += 1

                # --- Generate or reuse the synthetic CSV (cache-aware) ---
                fname = f"{variant_name}_{eps_label}_rep{rep}.csv"                  # synth-data filename
                fpath = os.path.join(SYNTH_DIR, fname)                              # full path
                if os.path.exists(fpath):                                           # cache: skip regeneration if already on disk
                    synth_df = pd.read_csv(fpath)                                   # load cached synth so logging still works
                    elapsed = 0.0                                                   # no compute spent
                else:                                                               # otherwise synthesise and persist
                    t0 = time.time()                                                # start timer
                    if eps == 'inf':                                                # near-noiseless baseline
                        synth_df = generate_no_dp_synthetic(df_variant, seed=seed)  # noiseless MST
                    else:                                                           # standard DP-MST
                        synth_df = generate_dp_synthetic(                           # DP-MST synthetic generation
                            df_variant, epsilon=float(eps), delta=DELTA, seed=seed)
                    elapsed = time.time() - t0                                      # record elapsed time
                    synth_df.to_csv(fpath, index=False)                             # persist for reproducibility

                # --- Log summary statistics for quality monitoring ---
                s_treat = synth_df['treatment'].mean()
                s_conv  = synth_df['conversion'].mean()
                s_visit = synth_df['visit'].mean()
                s_n     = synth_df.shape[0]

                results_log.append({
                    'variant':            variant_name,
                    'epsilon':            eps,
                    'rep':                rep,
                    'seed':               seed,
                    'n_original':         n_rows,
                    'n_synthetic':        s_n,
                    'treat_ratio_synth':  s_treat,
                    'conv_rate_synth':    s_conv,
                    'visit_rate_synth':   s_visit,
                    'time_seconds':       elapsed
                })

                # --- Progress indicator ---
                pct          = run_count / total_runs * 100
                elapsed_total = time.time() - start_time
                eta           = (elapsed_total / run_count) * (total_runs - run_count)
                print(f"  [{run_count}/{total_runs} ({pct:.0f}%)] "
                      f"{eps_label} rep{rep}: "
                      f"N={s_n:,} treat={s_treat:.3f} conv={s_conv:.4f} "
                      f"({elapsed:.1f}s, ETA {eta / 60:.0f}min)")

    # --- Save the synthesis log ---
    # This CSV is useful for diagnosing whether any conditions produced
    # distributions that deviate strongly from the original.
    log_df   = pd.DataFrame(results_log)
    log_path = os.path.join(SYNTH_DIR, "synthesis_log.csv")
    log_df.to_csv(log_path, index=False)

    total_time = time.time() - start_time
    print(f"\n{'=' * 70}")
    print(f"PHASE 2 COMPLETE")
    print(f"{'=' * 70}")
    print(f"Total runs: {run_count}")
    print(f"Total time: {total_time / 60:.1f} minutes")
    print(f"Synthesis log saved to: {log_path}")
    print(f"Synthetic datasets saved to: {SYNTH_DIR}")

    # Summary: mean conversion and treatment ratio per ε level
    print(f"\n--- Summary by epsilon ---")
    for eps in all_epsilons:
        sub = log_df[log_df['epsilon'] == eps]
        print(f"  eps={eps}: mean conv={sub['conv_rate_synth'].mean():.4f} "
              f"(sd={sub['conv_rate_synth'].std():.4f}), "
              f"mean treat={sub['treat_ratio_synth'].mean():.3f}")


PHASE 2: MST SYNTHESIS
Epsilon levels: [0.1, 0.5, 1.0, 2.0, 5.0, 10.0] + inf (no-DP baseline)
Delta: 1e-05
Repetitions per condition: 10
Benchmark variants: ['size_25', 'size_50', 'size_75', 'size_100', 'imbalance_baseline', 'imbalance_20', 'imbalance_05']

Total synthesis runs: 490

Variant: size_25 (N=11,200, treat=0.667, conv=0.0090)
  [1/490 (0%)] eps_0.1 rep0: N=11,176 treat=0.654 conv=0.0336 (0.0s, ETA 0min)
  [2/490 (0%)] eps_0.1 rep1: N=11,256 treat=0.664 conv=0.0126 (0.0s, ETA 0min)
  [3/490 (1%)] eps_0.1 rep2: N=11,115 treat=0.665 conv=0.0168 (0.0s, ETA 0min)
  [4/490 (1%)] eps_0.1 rep3: N=11,115 treat=0.665 conv=0.0015 (0.0s, ETA 0min)
  [5/490 (1%)] eps_0.1 rep4: N=11,029 treat=0.665 conv=0.0065 (0.0s, ETA 0min)
  [6/490 (1%)] eps_0.1 rep5: N=11,076 treat=0.678 conv=0.0352 (0.0s, ETA 0min)
  [7/490 (1%)] eps_0.1 rep6: N=11,207 treat=0.678 conv=0.0269 (0.0s, ETA 0min)
  [8/490 (2%)] eps_0.1 rep7: N=11,177 treat=0.643 conv=0.0000 (0.0s, ETA 0min)
  [9/490 (2%)] eps_0.1 rep8: 

In [6]:
"""
Phase 3 — Uplift Model Estimation & Evaluation
================================================
Steps 5–7 of the analysis procedure (Section 3.3):

  Step 5: Train the Two-Model uplift estimator on each synthetic training set;
          predict individual uplift scores on the real holdout test set.

  Step 6: Evaluate the quality of those predictions using three metrics:
            • Spearman ρ — ranking correlation with the oracle (H1, H2)
            • Qini coefficient — area-based uplift ranking measure (H1, H2)
            • AUUC — Area Under the Uplift Curve (additional robustness check)

  Step 7: Aggregate results across 10 repetitions per condition and save.

Input:  490 synthetic CSVs from Phase 2 (SYNTH_DIR)
        test.csv and size_100.csv from Phase 1 (DATA_DIR)
Output: results_phase3.csv (one row per run), results_summary.csv (aggregated)
"""

import os
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
import matplotlib
matplotlib.use('Agg')   # non-interactive backend — no display required
import matplotlib.pyplot as plt
import warnings
import time

warnings.filterwarnings('ignore')

# NumPy 2.0 renamed np.trapz to np.trapezoid; this handles both versions
_trapz = getattr(np, 'trapezoid', np.trapz)


# =============================================================================
# CONFIGURATION
# =============================================================================
# DATA_DIR, SYNTH_DIR, and RESULTS_DIR are defined in Cell 0.
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "qini_curves"), exist_ok=True)

# Must match Phase 2 exactly — the synthetic filenames encode these values
EPSILON_LEVELS = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
EPSILON_INF    = 1e6   # stand-in for ε = ∞ (no-DP baseline)
ALL_EPSILONS   = EPSILON_LEVELS + [EPSILON_INF]
EPS_LABELS     = {e: f"eps_{e}" for e in EPSILON_LEVELS}
EPS_LABELS[EPSILON_INF] = "eps_inf"

DELTA         = 1e-5
N_REPETITIONS = 10
BASE_SEED     = 42

SIZE_VARIANTS      = ['size_25', 'size_50', 'size_75', 'size_100']
IMBALANCE_VARIANTS = ['imbalance_baseline', 'imbalance_20', 'imbalance_05']
ALL_VARIANTS       = SIZE_VARIANTS + IMBALANCE_VARIANTS

# Seven pre-treatment features used by the uplift model (same as Phase 1)
FEATURES     = ['recency', 'history_segment', 'mens', 'womens', 'zip_code', 'newbie', 'channel']
CAT_FEATURES = ['history_segment', 'zip_code', 'channel']   # need one-hot encoding


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def encode_features(df, features=FEATURES, cat_features=CAT_FEATURES):
    """
    One-hot encode categorical features and return a float DataFrame.
    drop_first=True avoids perfect multicollinearity in logistic regression.
    The same encoding is applied to every dataset (train and test) so that
    column alignment is maintained — any missing dummies are filled with 0.
    """
    df_enc = df[features].copy()
    df_enc = pd.get_dummies(df_enc, columns=cat_features, drop_first=True)
    for col in df_enc.columns:
        df_enc[col] = df_enc[col].astype(float)
    return df_enc


def _fit_or_constant(X_train, y_train, X_test, label=""):
    """
    Fit logistic regression on (X_train, y_train) and return predicted
    probabilities for X_test.

    If y_train contains only one class (degenerate case — can occur in very
    small or heavily DP-noised synthetic datasets), return the observed class
    frequency as a constant rather than crashing. This is the 'n_degenerate'
    scenario tracked in the results.
    """
    n_classes = len(np.unique(y_train))
    if n_classes < 2:
        # Degenerate: all training labels are the same — no model can be fitted
        p = y_train.mean()
        return np.full(X_test.shape[0], p)

    model = LogisticRegression(max_iter=1000, solver='lbfgs', random_state=42, C=1.0)
    model.fit(X_train, y_train)
    return model.predict_proba(X_test)[:, 1]   # probability of class 1


def train_two_model_uplift(df_train, X_test_encoded, test_columns, outcome='visit'):
    """
    Two-Model (T-Learner) uplift estimator (Section 3.3.1):

    The Two-Model approach trains two separate response models:
      • Model T: P(visit = 1 | X, treatment = 1)  — fitted on treated rows
      • Model C: P(visit = 1 | X, treatment = 0)  — fitted on control rows

    Individual uplift is then estimated as the difference:
      τ̂(x) = P̂_T(x) − P̂_C(x)

    Primary outcome is visit (not conversion) — conversion is too sparse
    (~0.9% rate) to produce reliable uplift signals (Section 3.3).

    Also returns is_degenerate: True if either arm has only one class,
    meaning no meaningful model could be fitted (tracked as n_degenerate).

    Column alignment: synthetic data may have fewer dummy columns than the
    real test set (e.g., if a category is absent in the synthetic sample).
    Any missing columns are added as zero before scoring.
    """
    # Encode the synthetic training features (same pipeline as the test set)
    X_train = encode_features(df_train)

    # Align columns: add any test-set dummy columns missing from the synthetic set
    for col in test_columns:
        if col not in X_train.columns:
            X_train[col] = 0.0
    X_train = X_train[test_columns]   # enforce identical column order

    # Split into treated and control sub-samples
    treated_mask = df_train['treatment'] == 1
    control_mask = df_train['treatment'] == 0

    X_treated  = X_train[treated_mask]
    y_treated  = df_train.loc[treated_mask, outcome].values   # visit, not conversion
    X_control  = X_train[control_mask]
    y_control  = df_train.loc[control_mask, outcome].values   # visit, not conversion

    # Track degenerate cases: single class in either arm → no model can be fitted
    is_degenerate = (len(np.unique(y_treated)) < 2) or (len(np.unique(y_control)) < 2)

    # Fit the two models and subtract to get τ̂(x) for every test observation
    p_treated = _fit_or_constant(X_treated, y_treated, X_test_encoded, "treated")
    p_control = _fit_or_constant(X_control, y_control, X_test_encoded, "control")

    return p_treated - p_control, is_degenerate   # uplift scores + degenerate flag


def compute_qini_curve(uplift_scores, treatment, outcome, n_points=100):
    """
    Compute the Qini curve (Radcliffe, 2007).

    The Qini curve plots — as we target increasingly larger fractions of the
    population in descending uplift score order — the net incremental gain:
      Q(f) = ΣY_T / N_T_total − ΣY_C / N_C_total   (scaled by N)

    This normalises by the total number of treated and control units so that
    an unequal treatment ratio does not bias the curve.

    Returns: (fractions, qini_values) — arrays for plotting and AUC computation.
    """
    n               = len(uplift_scores)
    order           = np.argsort(-uplift_scores)   # descending: best targets first
    treatment_sorted = treatment[order]
    outcome_sorted   = outcome[order]

    fractions       = np.linspace(0, 1, n_points + 1)
    qini_values     = np.zeros(n_points + 1)

    n_treated_total = treatment.sum()
    n_control_total = n - n_treated_total

    for i, frac in enumerate(fractions):
        if frac == 0:
            continue   # Q(0) = 0 by definition
        k        = min(int(np.ceil(frac * n)), n)
        t_top    = treatment_sorted[:k]
        y_top    = outcome_sorted[:k]

        if n_treated_total > 0 and n_control_total > 0:
            qini_values[i] = (
                (y_top * t_top).sum() / n_treated_total -
                (y_top * (1 - t_top)).sum() / n_control_total
            )

    # Scale by N to convert rates back into counts (common convention)
    qini_values = qini_values * n
    return fractions, qini_values


def compute_qini_coefficient(fractions, qini_values):
    """
    Qini coefficient = area under the Qini curve minus the area under the
    random-targeting baseline.

    The random baseline is a straight line from Q(0)=0 to Q(1), so its area
    equals 0.5 × Q(1). A higher coefficient means better uplift ranking.
    """
    auc_qini   = _trapz(qini_values, fractions)    # area under the model curve
    auc_random = 0.5 * qini_values[-1]              # area under the diagonal
    return auc_qini - auc_random


def compute_auuc(uplift_scores, treatment, outcome, n_points=100):
    """
    Area Under the Uplift Curve (AUUC) — an alternative to the Qini coefficient.

    At each targeting fraction f, the uplift is estimated as the difference
    in conversion rates between targeted treated and targeted control units.
    AUUC is the integral of these point-wise uplift estimates over f ∈ (0, 1].
    A higher AUUC indicates the model more effectively identifies high-uplift
    individuals.
    """
    n               = len(uplift_scores)
    order           = np.argsort(-uplift_scores)
    treatment_sorted = treatment[order]
    outcome_sorted   = outcome[order]

    fractions       = np.linspace(0, 1, n_points + 1)[1:]   # exclude f=0
    uplift_values   = np.zeros(len(fractions))

    for i, frac in enumerate(fractions):
        k    = min(int(np.ceil(frac * n)), n)
        t_top = treatment_sorted[:k]
        y_top  = outcome_sorted[:k]
        n_t   = t_top.sum()
        n_c   = k - n_t

        if n_t > 0 and n_c > 0:
            uplift_values[i] = (
                (y_top * t_top).sum() / n_t -
                (y_top * (1 - t_top)).sum() / n_c
            )

    return _trapz(uplift_values, fractions)


# =============================================================================
# MAIN EVALUATION LOOP
# =============================================================================

if __name__ == "__main__":
    print("=" * 70)
    print("PHASE 3: UPLIFT MODEL ESTIMATION & EVALUATION")
    print("=" * 70)

    # --- Load the real holdout test set (never used in training) ---
    df_test      = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
    X_test       = encode_features(df_test)
    test_columns = X_test.columns.tolist()   # column order fixed for all subsequent runs
    treatment_arr = df_test['treatment'].values
    outcome_arr   = df_test['visit'].values     # primary outcome: visit (Section 3.3)

    # --- Oracle baseline: Two-Model trained on the real (non-synthetic) size_100 data ---
    # The oracle represents the upper bound — the best the Two-Model estimator
    # can achieve when trained on real data. All synthetic conditions are
    # evaluated relative to this oracle.
    df_train_real    = pd.read_csv(os.path.join(DATA_DIR, "size_100.csv"))
    oracle_scores, _ = train_two_model_uplift(df_train_real, X_test, test_columns, outcome='visit')
    frac_oracle, qini_oracle = compute_qini_curve(oracle_scores, treatment_arr, outcome_arr)
    qini_coef_oracle = compute_qini_coefficient(frac_oracle, qini_oracle)
    auuc_oracle      = compute_auuc(oracle_scores, treatment_arr, outcome_arr)
    print(f"Oracle: Qini={qini_coef_oracle:.4f}, AUUC={auuc_oracle:.6f}")

    # --- Main loop: evaluate every synthetic dataset ---
    total_conditions = len(ALL_VARIANTS) * len(ALL_EPSILONS) * N_REPETITIONS
    results   = []
    run_count = 0
    t_start   = time.time()

    for variant_name in ALL_VARIANTS:
        for eps in ALL_EPSILONS:
            eps_label   = EPS_LABELS[eps]
            eps_display = 'inf' if eps == EPSILON_INF else str(eps)

            for rep in range(N_REPETITIONS):
                # Reconstruct the filename that Phase 2 used to save this synthetic set
                fname = f"{variant_name}_{eps_label}_rep{rep}.csv"
                fpath = os.path.join(SYNTH_DIR, fname)

                if not os.path.exists(fpath):
                    continue   # skip if Phase 2 did not produce this file

                run_count += 1
                df_synth = pd.read_csv(fpath)

                # --- Step 5: Train on synthetic data, predict on real test set ---
                synth_scores, is_degen = train_two_model_uplift(
                    df_synth, X_test, test_columns, outcome='visit'
                )

                # --- Step 6a: Spearman ρ — ranking correlation with oracle ---
                # ρ = 1 means perfect preservation of the oracle ranking;
                # ρ ≈ 0 or negative means privacy noise destroyed the ordering.
                if synth_scores.std() == 0:
                    # Degenerate: all uplift scores are identical — no ranking
                    rho, p_val = np.nan, np.nan
                else:
                    rho, p_val = spearmanr(oracle_scores, synth_scores)

                # --- Step 6b: Qini coefficient and AUUC ---
                frac_s, qini_s = compute_qini_curve(synth_scores, treatment_arr, outcome_arr)
                qini_coef      = compute_qini_coefficient(frac_s, qini_s)
                auuc           = compute_auuc(synth_scores, treatment_arr, outcome_arr)

                # --- Step 7: Record results for this run ---
                results.append({
                    'variant':           variant_name,
                    'epsilon':           eps_display,
                    'rep':               rep,
                    'spearman_rho':      rho,
                    'spearman_p':        p_val,
                    'qini_coef':         qini_coef,
                    'auuc':              auuc,
                    'n_synth':           df_synth.shape[0],
                    'treat_ratio_synth': df_synth['treatment'].mean(),
                    'conv_rate_synth':   df_synth['conversion'].mean(),
                    'is_degenerate':     int(is_degen),   # 1 if model could not be fitted
                })

                # Progress print every 50 runs
                if run_count % 50 == 0:
                    elapsed = time.time() - t_start
                    eta     = (elapsed / run_count) * (total_conditions - run_count)
                    print(f"  [{run_count}/{total_conditions}] ETA {eta / 60:.0f}m")

    # --- Save individual-run results ---
    results_df = pd.DataFrame(results)
    results_df.to_csv(os.path.join(RESULTS_DIR, "results_phase3.csv"), index=False)

    # --- Aggregate across repetitions: mean and SD for each metric ---
    # These condition-level summaries are what Phase 4 uses for tables and figures.
    summary = results_df.groupby(['variant', 'epsilon']).agg(
        rho_mean    =('spearman_rho',  'mean'),
        rho_sd      =('spearman_rho',  'std'),
        qini_mean   =('qini_coef',     'mean'),
        qini_sd     =('qini_coef',     'std'),
        auuc_mean   =('auuc',          'mean'),
        auuc_sd     =('auuc',          'std'),
        n_reps      =('rep',           'count'),
        n_degenerate=('is_degenerate', 'sum')   # count of runs where model could not be fitted
    ).reset_index()
    summary.to_csv(os.path.join(RESULTS_DIR, "results_summary.csv"), index=False)

    print(f"\nPhase 3 complete. {run_count} runs in {(time.time() - t_start) / 60:.1f} minutes.")


PHASE 3: UPLIFT MODEL ESTIMATION & EVALUATION
Oracle: Qini=66.5221, AUUC=0.060743
  [50/490] ETA 1m
  [100/490] ETA 1m
  [150/490] ETA 1m
  [200/490] ETA 1m
  [250/490] ETA 1m
  [300/490] ETA 1m
  [350/490] ETA 1m
  [400/490] ETA 0m
  [450/490] ETA 0m

Phase 3 complete. 490 runs in 2.0 minutes.


# Per-arm response-surface diagnostic for §4.4.2 negative-ρ finding.

In [7]:
# =============================================================================
# Per-arm response-surface diagnostic for §4.4.2 negative-ρ finding.
# Splits the size_25 / size_50 anti-correlated uplift ρ into its two
# components: P(visit | X, T=1) and P(visit | X, T=0). If only one of the two
# is anti-correlated with the oracle, the flip is localised to that arm.
# =============================================================================

import os                                                # synthetic file paths
import numpy as np                                       # NaN-safe aggregation
import pandas as pd                                      # DataFrame output
from scipy.stats import spearmanr                        # ρ between response surfaces

# --- 1. Oracle per-arm response surfaces (real size_100, no DP, no MST) ---
df_real_100   = pd.read_csv(os.path.join(DATA_DIR, "size_100.csv"))   # real reference training set
X_oracle      = encode_features(df_real_100)                          # one-hot encode features
for col in test_columns:                                              # column-align with test set
    if col not in X_oracle.columns:
        X_oracle[col] = 0.0                                           # missing dummy → 0
X_oracle      = X_oracle[test_columns]                                # enforce column order

mask_t_real   = df_real_100['treatment'] == 1                         # treated rows in real data
mask_c_real   = df_real_100['treatment'] == 0                         # control rows in real data

oracle_p_t    = _fit_or_constant(                                     # P̂(visit=1 | X, T=1) on holdout
                    X_oracle[mask_t_real],
                    df_real_100.loc[mask_t_real, 'visit'].values,
                    X_test, "oracle_t")
oracle_p_c    = _fit_or_constant(                                     # P̂(visit=1 | X, T=0) on holdout
                    X_oracle[mask_c_real],
                    df_real_100.loc[mask_c_real, 'visit'].values,
                    X_test, "oracle_c")

# --- 2. Synthetic per-arm response surfaces at ε=∞ for each size variant ---
diagnostic_rows = []                                                  # one row per variant
for variant in ['size_25', 'size_50', 'size_75', 'size_100']:         # 25/50 = problem; 75/100 = controls
    rho_t_list, rho_c_list = [], []                                   # accumulate per-rep ρ
    for rep in range(N_REPETITIONS):                                  # 10 reps as in the main analysis
        fpath = os.path.join(SYNTH_DIR, f"{variant}_eps_inf_rep{rep}.csv")  # ε=∞ synthetic file
        if not os.path.exists(fpath):
            continue                                                  # skip missing reps
        df_s  = pd.read_csv(fpath)                                    # one synthetic dataset

        X_s   = encode_features(df_s)                                 # encode synthetic features
        for col in test_columns:                                      # align columns with test set
            if col not in X_s.columns:
                X_s[col] = 0.0
        X_s   = X_s[test_columns]                                     # enforce column order

        mt    = df_s['treatment'] == 1                                # synthetic treated mask
        mc    = df_s['treatment'] == 0                                # synthetic control mask

        synth_p_t = _fit_or_constant(                                 # synthetic P̂_T surface on holdout
                        X_s[mt], df_s.loc[mt, 'visit'].values,
                        X_test, "synth_t")
        synth_p_c = _fit_or_constant(                                 # synthetic P̂_C surface on holdout
                        X_s[mc], df_s.loc[mc, 'visit'].values,
                        X_test, "synth_c")

        # ρ against the oracle per-arm surface — the actual diagnostic
        rho_t = spearmanr(oracle_p_t, synth_p_t).correlation if synth_p_t.std() > 0 else np.nan
        rho_c = spearmanr(oracle_p_c, synth_p_c).correlation if synth_p_c.std() > 0 else np.nan
        rho_t_list.append(rho_t)                                      # collect for variant aggregation
        rho_c_list.append(rho_c)

    # Aggregate across the 10 reps
    diagnostic_rows.append({
        'variant':            variant,
        'rho_treated_mean':   np.nanmean(rho_t_list),                 # treatment-arm preservation
        'rho_treated_sd':     np.nanstd(rho_t_list, ddof=1),
        'rho_control_mean':   np.nanmean(rho_c_list),                 # control-arm preservation
        'rho_control_sd':     np.nanstd(rho_c_list, ddof=1),
        'n_reps_used':        sum(1 for r in rho_t_list if not np.isnan(r)),
    })

# --- 3. Save and print ---
diag_df = pd.DataFrame(diagnostic_rows)

analysis_dir = os.path.join(
    os.path.dirname(os.path.dirname(os.path.dirname(RESULTS_DIR))),
    "extra_analysis",
    "negative_rho_diagnosis_data_size"
)

os.makedirs(analysis_dir, exist_ok=True)

diag_df.to_csv(
    os.path.join(analysis_dir, "negative_rho_diagnosis_data_size.csv"),
    index=False
)

print(diag_df.to_string(index=False))

 variant  rho_treated_mean  rho_treated_sd  rho_control_mean  rho_control_sd  n_reps_used
 size_25         -0.324241        0.036583         -0.345833        0.025834           10
 size_50         -0.167034        0.015568         -0.132093        0.012013           10
 size_75          0.587511        0.007587          0.621585        0.002519           10
size_100          0.592239        0.006421          0.619750        0.004059           10


In [8]:
# Appendix Table A11 — Real-data size-variant baseline (no MST, no DP). (thesis Table A11)
# Run after cell 2; reuses oracle_scores, X_test, test_columns,
# treatment_arr, outcome_arr, and the Phase-3 helpers.
a11_rows = []
for variant in ['size_25', 'size_50', 'size_75', 'size_100']:
    df_real = pd.read_csv(os.path.join(DATA_DIR, f"{variant}.csv"))
    scores, degen = train_two_model_uplift(df_real, X_test, test_columns, outcome='visit')
    rho, _ = spearmanr(oracle_scores, scores)
    frac, q = compute_qini_curve(scores, treatment_arr, outcome_arr)
    a11_rows.append({'variant': variant,
                    'n_train': len(df_real),
                    'rho_real': rho,
                    'qini_real': compute_qini_coefficient(frac, q),
                    'degenerate': int(degen)})

appendix_a11 = pd.DataFrame(a11_rows)
appendix_a11.to_csv(os.path.join(RESULTS_DIR, "appendix_table_a11_real_baseline.csv"), index=False)
print(appendix_a11.to_string(index=False))

 variant  n_train  rho_real  qini_real  degenerate
 size_25    11200  0.834050  69.344201           0
 size_50    22400  0.914317  40.321307           0
 size_75    33600  0.967885  61.999561           0
size_100    44800  1.000000  66.522097           0


In [9]:
# Appendix Table A13 — Subgroup-imbalance per-condition results: rho, Qini, degenerate runs by privacy level. (thesis Table A13)
# Mirrors A1/A2 layout; one row per (variant, ε) cell.
import os, numpy as np, pandas as pd

summary = pd.read_csv(os.path.join(RESULTS_DIR, "results_summary.csv"))

LABELS = {'imbalance_baseline': 'Baseline (50% newbie)',
          'imbalance_20':       '20% newbie',
          'imbalance_05':       '5% newbie'}
ORDER  = ['imbalance_baseline', 'imbalance_20', 'imbalance_05']

a13 = summary[summary['variant'].isin(ORDER)].copy()
a13['variant'] = pd.Categorical(a13['variant'], ORDER, ordered=True)
a13 = a13.sort_values(['variant', 'epsilon'])

a13_out = pd.DataFrame({
    'Variant':         a13['variant'].map(LABELS).values,
    'ε':               a13['epsilon'].replace(np.inf, '∞').astype(str).values,
    'Mean ρ':          a13['rho_mean'].round(4).values,
    'SD ρ':            a13['rho_sd'].round(4).values,
    'Mean Qini':       a13['qini_mean'].round(2).values,
    'SD Qini':         a13['qini_sd'].round(2).values,
    'Degenerate runs': a13['n_degenerate'].astype(str).str.cat(['/10'] * len(a13)).values,
})

a13_out.to_csv(os.path.join(RESULTS_DIR, "appendix_table_a13_subgroup_imbalance_results.csv"),
              index=False)
print(a13_out.to_string(index=False))

              Variant    ε  Mean ρ   SD ρ  Mean Qini  SD Qini Degenerate runs
Baseline (50% newbie)  0.1  0.1232 0.1876      -0.95    14.70            2/10
Baseline (50% newbie)  0.5  0.3713 0.0239       6.98     6.33            0/10
Baseline (50% newbie)  1.0  0.3761 0.0202       9.95     3.88            0/10
Baseline (50% newbie)  2.0  0.3814 0.0117       9.66     2.74            0/10
Baseline (50% newbie)  5.0  0.3833 0.0057      11.09     1.28            0/10
Baseline (50% newbie) 10.0  0.3849 0.0032      11.28     1.08            0/10
Baseline (50% newbie)    ∞  0.3839 0.0022      10.97     1.69            0/10
           20% newbie  0.1  0.1140 0.1364      -9.42    17.44            3/10
           20% newbie  0.5  0.2439 0.1769      19.70    22.69            0/10
           20% newbie  1.0  0.3346 0.1737      31.28    21.45            0/10
           20% newbie  2.0  0.4003 0.1217      40.68    17.67            0/10
           20% newbie  5.0  0.4246 0.0386      41.28     5.71   

In [10]:
# =============================================================================
# Exploratory ε-elbow analysis for size_100 (Plan v2, Section B).
# Generates thesis Figure A1 (elbow plot) and Table A9 (elbow summary).
# Adds ε = 0.2, 0.3, 0.4 to localise the practical privacy-budget elbow ε*.
# =============================================================================
import os                                                                          # file-path handling
import numpy as np                                                                 # numerical ops and np.inf
import pandas as pd                                                                # dataframes and CSV I/O
import matplotlib.pyplot as plt                                                    # ε-elbow plot
from scipy.stats import spearmanr                                                  # Spearman ranking preservation

# --- Settings ---
VARIANT, NEW_EPS, REPS = "size_100", [0.2, 0.3, 0.4], 10                           # condition, new ε grid, reps
SEED0 = globals().get("BASE_SEED", 42)                                             # reuse thesis seed if set, else 42
OUT_DIR = ELBOW_SYNTH_DIR                                                          # subfolder for the new synthetic files
os.makedirs(OUT_DIR, exist_ok=True)                                                # create the folder if missing
runs_csv = os.path.join(ELBOW_RESULTS_DIR, "results_exploratory_epsilon_elbow_runs.csv") # run-level output
sum_csv  = os.path.join(ELBOW_RESULTS_DIR, "summary_exploratory_epsilon_elbow.csv")      # appendix-ready summary
dec_csv  = os.path.join(ELBOW_RESULTS_DIR, "decision_exploratory_epsilon_elbow.csv")     # ε* decision record
fig_png  = os.path.join(ELBOW_FIGURES_DIR, "figure_exploratory_epsilon_elbow_size100.png")  # saved figure

# --- Reference uplift ranking from real size_100 data ---
train  = pd.read_csv(os.path.join(DATA_DIR, f"{VARIANT}.csv"))                     # real size_100 training partition
test   = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))                           # unchanged real holdout
X_test = encode_features(test)                                                     # encode predictors as in main analysis
test_cols = X_test.columns.tolist()                                                # store column order for model consistency
treat, y  = test["treatment"].values, test["visit"].values                         # treatment indicator and primary outcome
ref_scores, _ = train_two_model_uplift(train, X_test, test_cols, outcome="visit")  # clean reference ranking on holdout

# --- Run new ε values on size_100 ---
rows = []                                                                          # collect one row per (ε, rep)
for eps in NEW_EPS:                                                                # loop over new ε
    for rep in range(REPS):                                                        # 10 reps each
        fpath = os.path.join(OUT_DIR, f"{VARIANT}_eps_{eps}_rep{rep}.csv")         # synthetic-data path for this run
        if os.path.exists(fpath):                                                  # cache: do not regenerate if cached
            synth = pd.read_csv(fpath)                                             # reuse cached synthetic data
        else:                                                                      # otherwise synthesise and cache
            synth = generate_dp_synthetic(train, epsilon=eps, delta=DELTA,         # DP-MST synthetic generation
                                          seed=SEED0 + rep)                        # deterministic per-rep seed
            synth.to_csv(fpath, index=False)                                       # persist for reproducibility
        scores, degen = train_two_model_uplift(synth, X_test, test_cols,           # train on synthetic, score real holdout
                                               outcome="visit")
        rho = spearmanr(ref_scores, scores).correlation if np.std(scores) > 0 else np.nan  # ρ vs reference; NaN if flat
        frac, qcurve = compute_qini_curve(scores, treat, y)                        # build Qini curve on real holdout
        rows.append({"variant": VARIANT, "epsilon": eps, "rep": rep, "rho": rho,   # store run-level metrics
                     "qini": compute_qini_coefficient(frac, qcurve),
                     "auuc": compute_auuc(scores, treat, y),
                     "degenerate": int(degen)})
runs = pd.DataFrame(rows)                                                          # run-level dataframe
runs.to_csv(runs_csv, index=False)                                                 # save run-level results

# --- Summarise new ε values and combine with existing main-result anchors ---
new_summary = runs.groupby(["variant", "epsilon"]).agg(                            # aggregate new runs to (ε) cells
    rho_mean=("rho", "mean"),   rho_sd=("rho", "std"),                             # ρ central tendency and spread
    qini_mean=("qini", "mean"), qini_sd=("qini", "std"),                           # Qini central tendency and spread
    auuc_mean=("auuc", "mean"), auuc_sd=("auuc", "std"),                           # AUUC central tendency and spread
    n_reps=("rep", "count"),    n_degenerate=("degenerate", "sum"),                # rep count and degenerate count
).reset_index()                                                                    # back to flat dataframe
main = pd.read_csv(os.path.join(RESULTS_DIR, "results_summary.csv"))               # main thesis summary results
main["epsilon"] = main["epsilon"].apply(                                           # normalise ε column to numeric, incl. ∞
    lambda x: np.inf if str(x).lower() in {"inf", "infinity", "∞"} else float(x))
anchor_eps = [0.1, 0.5, 1, 2, 5, 10, np.inf]                                       # original ε anchors
anchors = main[(main["variant"] == VARIANT) &                                      # filter to size_100 anchors
               main["epsilon"].isin(anchor_eps)][new_summary.columns]              # keep matching columns
summary = pd.concat([anchors, new_summary], ignore_index=True) \
            .sort_values("epsilon").reset_index(drop=True)                         # combine and order strict→weak privacy

# --- Plateau rule and ε* selection ---
plateau = summary[summary["epsilon"].isin([0.5, 1, 2, 5, 10, np.inf])]             # plateau region (higher-ε anchors)
rho_plateau, sd_plateau = plateau["rho_mean"].mean(), plateau["rho_sd"].median()   # plateau centre; typical SD
rho_cut, sd_cut = 0.90 * rho_plateau, 2 * sd_plateau                               # 90%-of-plateau ρ; SD-stabilisation cut
cand = summary[summary["epsilon"].isin([0.1, 0.2, 0.3, 0.4, 0.5])].copy()          # candidate elbow region
cand["pass_rule"] = ((cand["rho_mean"]    >= rho_cut) &                            # ρ within 10% of plateau
                     (cand["rho_sd"]      <= sd_cut)  &                            # SD has stabilised
                     (cand["n_degenerate"] <= 1))                                  # synthesis reliably non-degenerate
passed = cand[cand["pass_rule"]].sort_values("epsilon")                            # smallest-ε first
epsilon_star = passed["epsilon"].iloc[0] if len(passed) else 0.5                   # smallest passing ε; fallback to 0.5
decision = pd.DataFrame([{"variant": VARIANT, "rho_plateau": rho_plateau,          # decision record for thesis reporting
                          "sd_plateau": sd_plateau, "rho_cutoff_90pct": rho_cut,
                          "sd_cutoff_2x": sd_cut, "epsilon_star": epsilon_star,
                          "fallback_used": len(passed) == 0}])
summary["epsilon_label"] = summary["epsilon"].apply(                               # readable ε labels including ∞
    lambda v: "∞" if np.isinf(v) else f"{v:g}")
summary.to_csv(sum_csv, index=False)                                               # save combined ε summary
decision.to_csv(dec_csv, index=False)                                              # save ε* decision

# --- Figure: ε-elbow curve ---
x = np.arange(len(summary))                                                        # categorical x positions for readability
plt.figure(figsize=(8, 5))                                                         # figure size
plt.errorbar(x, summary["rho_mean"], yerr=summary["rho_sd"],                       # mean ρ with ±1 SD bars
             marker="o", capsize=4)
plt.axhline(rho_cut, linestyle="--", linewidth=1, label="90% of plateau ρ")        # 90%-of-plateau decision line
if (summary["epsilon"] == epsilon_star).any():                                     # mark ε* if it is on the grid
    plt.axvline(int(np.where(summary["epsilon"] == epsilon_star)[0][0]),           # x-position of ε*
                linestyle=":", linewidth=1, label=f"ε* = {epsilon_star:g}")
plt.xticks(x, summary["epsilon_label"])                                            # tick labels including ∞
plt.xlabel("Privacy budget (ε)"); plt.ylabel("Spearman ρ (mean ± SD)")             # axis labels
plt.title("Exploratory ε-elbow analysis for size_100"); plt.legend()               # title and legend
plt.tight_layout(); plt.savefig(fig_png, dpi=300); plt.show()                      # save and display

# --- Console output ---
print("ε-elbow summary")                                                           # header
print(summary[["epsilon_label", "rho_mean", "rho_sd",                              # thesis-ready overview
               "qini_mean", "qini_sd", "n_degenerate"]].to_string(index=False))
print("\nDecision"); print(decision.to_string(index=False))                        # ε* and thresholds

ε-elbow summary
epsilon_label  rho_mean   rho_sd  qini_mean   qini_sd  n_degenerate
          0.1  0.123173 0.187621  -0.951699 14.700977             2
          0.2  0.164990 0.192068  17.752145 26.492551             0
          0.3  0.228031 0.162472  10.796312 25.582333             0
          0.4  0.354252 0.070126  12.799494 14.583980             0
          0.5  0.371301 0.023871   6.979216  6.331245             0
            1  0.376121 0.020246   9.948327  3.881373             0
            2  0.381416 0.011696   9.657375  2.738392             0
            5  0.383255 0.005665  11.091191  1.277985             0
           10  0.384872 0.003229  11.283403  1.084339             0
            ∞  0.383901 0.002172  10.973812  1.689264             0

Decision
 variant  rho_plateau  sd_plateau  rho_cutoff_90pct  sd_cutoff_2x  epsilon_star  fallback_used
size_100     0.380144     0.00868           0.34213      0.017361           0.5           True


In [11]:
#!/usr/bin/env python3
"""
Phase 4 — Figures & Tables for Chapter 4
=========================================
Produces the figures and printed tables for Sections 4.4-4.5 and the
hypothesis-evaluation statistics for Section 4.7. Section 4.6 (separate
e-mail treatment robustness) is produced by the pooling/split extension cell.
Formatting follows APA 7th Edition (§7.26–7.36): Times New Roman, no gridlines,
open frame (top + right spines removed), legend inside without border.

Input:  results_summary.csv, results_phase3.csv  (from Phase 3)
Output: Five PNG figures (300 DPI) saved to FIGURES_DIR
        Printed Tables 7 and A12 (copy into thesis document)
"""

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.ticker as mticker
from scipy.stats import spearmanr, pearsonr
import os
import subprocess
import math
import warnings
warnings.filterwarnings('ignore')


# =============================================================================
# PATHS
# =============================================================================
# RESULTS_DIR and FIGURES_DIR are defined in Cell 0.
os.makedirs(FIGURES_DIR, exist_ok=True)

# =============================================================================
# FONT SETUP — Times New Roman (APA 7 body-text font)
# =============================================================================
def _setup_times_new_roman():
    """
    Detect Times New Roman on this machine and return its font name.
    If not found (e.g., on a fresh Linux install), fall back to STIXGeneral,
    which is metrically near-identical and renders properly in matplotlib.
    """
    fm._load_fontmanager(try_read_cache=False)
    available = {f.name for f in fm.fontManager.ttflist}

    if 'Times New Roman' in available:
        return 'Times New Roman'

    # macOS sometimes needs a cache rebuild after an Office install
    try:
        subprocess.run(['fc-list', ':family=Times New Roman'],
                       capture_output=True, check=True)
        fm._load_fontmanager(try_read_cache=False)
        available = {f.name for f in fm.fontManager.ttflist}
        if 'Times New Roman' in available:
            return 'Times New Roman'
    except Exception:
        pass

    print("⚠  Times New Roman not found — using STIX (metrically similar serif)")
    return 'STIXGeneral'

TNR = _setup_times_new_roman()


# =============================================================================
# APA 7TH EDITION GLOBAL PLOT STYLE (§7.26–7.36)
# =============================================================================
# Applied globally so every figure inherits these settings automatically.
# Key APA rules enforced here:
#   • No bold axis labels or tick labels (only figure numbers are bold)
#   • No gridlines
#   • Top and right spines removed (open frame)
#   • Legend: no border, no background fill
#   • Minimum 300 DPI for print output
plt.rcParams.update({
    # Font
    'font.family':        'serif',
    'font.serif':         [TNR, 'Times New Roman', 'Times', 'STIX', 'DejaVu Serif'],
    'font.size':          11,
    'mathtext.fontset':   'stix',           # math symbols match TNR

    # Figure dimensions and background
    'figure.dpi':         150,              # screen preview
    'figure.facecolor':   'white',
    'figure.edgecolor':   'white',
    'savefig.dpi':        300,              # APA: ≥ 300 DPI
    'savefig.bbox':       'tight',
    'savefig.facecolor':  'white',
    'savefig.edgecolor':  'white',
    'savefig.pad_inches': 0.15,

    # Axes
    'axes.facecolor':     'white',
    'axes.edgecolor':     'black',
    'axes.linewidth':     0.8,
    'axes.grid':          False,            # APA: no gridlines
    'axes.spines.top':    False,            # APA: open frame
    'axes.spines.right':  False,
    'axes.labelsize':     12,
    'axes.labelweight':   'normal',         # APA: no bold axis labels
    'axes.titlesize':     12,
    'axes.titleweight':   'normal',

    # Ticks
    'xtick.labelsize':       10,
    'ytick.labelsize':       10,
    'xtick.direction':       'out',
    'ytick.direction':       'out',
    'xtick.major.size':      4,
    'ytick.major.size':      4,
    'xtick.major.width':     0.8,
    'ytick.major.width':     0.8,
    'xtick.minor.visible':   False,
    'ytick.minor.visible':   False,

    # Legend
    'legend.fontsize':   10,
    'legend.frameon':    False,             # APA: no legend border
    'legend.fancybox':   False,
    'legend.shadow':     False,
    'legend.loc':        'best',

    # Lines
    'lines.linewidth':   1.5,
    'lines.markersize':  6,
})


# =============================================================================
# LOAD RESULTS DATA
# =============================================================================
summary = pd.read_csv(os.path.join(RESULTS_DIR, "results_summary.csv"))
detail  = pd.read_csv(os.path.join(RESULTS_DIR, "results_phase3.csv"))

# Convert epsilon from string to numeric so rows can be sorted and filtered
# 'inf' → float('inf'); we replace with 1e6 to match the Phase 2/3 convention
summary['eps_num'] = summary['epsilon'].astype(float).replace(np.inf, 1e6)
detail['eps_num']  = detail['epsilon'].astype(float).replace(np.inf, 1e6)

# Canonical epsilon ordering for x-axis ticks (lowest DP → no DP)
EPS_ORDER  = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 1e6]
EPS_LABELS = ['0.1', '0.5', '1', '2', '5', '10', '∞']

SIZE_VARIANTS = ['size_25', 'size_50', 'size_75', 'size_100']
IMB_VARIANTS  = ['imbalance_baseline', 'imbalance_20', 'imbalance_05']

# --- Colour palette: APA-friendly, distinguishable in greyscale print ---
SIZE_COLORS = {
    'size_25':   '#D62728',
    'size_50':   '#FF7F0E',
    'size_75':   '#2CA02C',
    'size_100':  '#1F77B4'
}
SIZE_LABELS = {
    'size_25':   '25%',
    'size_50':   '50%',
    'size_75':   '75%',
    'size_100':  '100%'
}

IMB_COLORS = {
    'imbalance_baseline': '#1F77B4',
    'imbalance_20':       '#FF7F0E',
    'imbalance_05':       '#D62728'
}
IMB_LABELS = {
    'imbalance_baseline': 'Baseline (50%)',
    'imbalance_20':       '20% newbie',
    'imbalance_05':       '5% newbie'
}

# Distinct markers so lines are separable in black-and-white print
SIZE_MARKERS = {'size_25': 'v', 'size_50': 's', 'size_75': '^', 'size_100': 'o'}
IMB_MARKERS  = {'imbalance_baseline': 'o', 'imbalance_20': 's', 'imbalance_05': 'v'}


# =============================================================================
# HELPER: APA 7 figure label + title (§7.26)
# =============================================================================
def apa_figure_title(fig, fig_number, title_text, note_text=None):
    """
    Add APA 7 compliant figure header above the plot area:
      Line 1 — "Figure X"  (bold, flush left)
      Line 2 — Descriptive title  (italic, flush left)
    Optionally add a Note below the figure (§7.28): italic, flush left.

    Coordinates use fig.transFigure (0–1 relative to the figure).
    """
    # "Figure X" — bold, flush left, just above the plot
    fig.text(0.02, 1.04, f'Figure {fig_number}',
             fontsize=12, fontweight='bold', fontstyle='normal',
             fontfamily='serif', va='bottom', transform=fig.transFigure)

    # Descriptive title — italic, flush left, one line below the label
    fig.text(0.02, 0.99, title_text,
             fontsize=12, fontweight='normal', fontstyle='italic',
             fontfamily='serif', va='bottom', transform=fig.transFigure)

    # Optional note below the figure
    if note_text:
        fig.text(0.02, -0.04, f'Note. {note_text}',
                 fontsize=10, fontweight='normal', fontstyle='italic',
                 fontfamily='serif', va='top', transform=fig.transFigure)


# =============================================================================
# HELPER: degradation-curve plot (reusable for Figures 1–4)
# =============================================================================
def plot_degradation(variants, metric, ylabel, fig_number, title_text,
                     colors, labels, markers, filename,
                     hline=None, hline_label=None, note=None):
    """
    Plot a performance metric (ρ or Qini) against ε for a set of variants.

    Each variant becomes one line; error bars show ± 1 SD across the 10
    Monte Carlo repetitions. X-axis positions are integer indices mapped to
    EPS_LABELS so the unequal spacing of epsilon levels is displayed linearly.

    Parameters
    ----------
    variants    : list of variant name strings to plot
    metric      : 'rho' or 'qini' — used to look up {metric}_mean / {metric}_sd
    ylabel      : y-axis label string
    fig_number  : APA figure number (printed in bold above the figure)
    title_text  : APA figure title (printed in italic)
    colors/labels/markers : per-variant dicts
    filename    : output PNG filename (saved to FIGURES_DIR)
    hline       : optional float — draws a horizontal reference line
    hline_label : label for the reference line
    note        : APA §7.28 figure note text
    """
    fig, ax = plt.subplots(figsize=(7.5, 5))

    for var in variants:
        df_var = summary[summary['variant'] == var].sort_values('eps_num')
        x_pos  = list(range(len(EPS_ORDER)))
        y_vals, y_errs = [], []

        for eps in EPS_ORDER:
            row = df_var[df_var['eps_num'] == eps]
            if len(row) > 0:
                y_vals.append(row[f'{metric}_mean'].values[0])
                y_errs.append(row[f'{metric}_sd'].values[0])
            else:
                y_vals.append(np.nan)
                y_errs.append(0)

        ax.errorbar(x_pos, y_vals, yerr=y_errs,
                    marker=markers[var], markersize=6, capsize=3,
                    label=labels[var], color=colors[var], linewidth=1.5,
                    markerfacecolor='white', markeredgecolor=colors[var],
                    markeredgewidth=1.5)

    # Optional horizontal reference line (e.g., oracle level)
    if hline is not None:
        ax.axhline(y=hline, color='black', linestyle='--', alpha=0.4, linewidth=0.8)
        if hline_label:
            ax.text(len(EPS_ORDER) - 0.5, hline + 0.02, hline_label,
                    fontsize=9, ha='right', va='bottom', alpha=0.6, fontstyle='italic')

    # Subtle zero-reference so readers can spot negative Qini conditions
    ax.axhline(y=0, color='black', linestyle='-', alpha=0.15, linewidth=0.6)

    ax.set_xticks(range(len(EPS_LABELS)))
    ax.set_xticklabels(EPS_LABELS)
    ax.set_xlabel('Privacy budget (ε)')
    ax.set_ylabel(ylabel)

    # Legend: APA — inside plot, no border, no fill
    leg = ax.legend(loc='best')
    leg.get_frame().set_linewidth(0)
    leg.get_frame().set_facecolor('none')

    apa_figure_title(fig, fig_number, title_text, note_text=note)
    plt.subplots_adjust(top=0.86, bottom=0.12)

    path = os.path.join(FIGURES_DIR, filename)
    plt.savefig(path)
    plt.show()
    print(f"Saved: {path}")


# =============================================================================
# FIGURE 3: Spearman ρ vs ε — Dataset-size block (H4)
# Answers: does larger training data protect against DP degradation?
# =============================================================================
print("=" * 60)
print("FIGURE 3: Ranking preservation by dataset size")
print("=" * 60)
plot_degradation(
    SIZE_VARIANTS, 'rho',
    ylabel='Spearman ρ (mean ± SD)',
    fig_number=3,
    title_text='Treatment-Effect Ranking Preservation Across Privacy Levels by Dataset Size',
    colors=SIZE_COLORS, labels=SIZE_LABELS, markers=SIZE_MARKERS,
    filename='fig3_rho_by_size.png',
    note='Error bars represent ± 1 SD across 10 Monte Carlo replications.'
)

# =============================================================================
# FIGURE 4: Spearman ρ vs ε — Subgroup-imbalance block (H5)
# Answers: does a rarer newbie subgroup amplify DP degradation?
# =============================================================================
print("\n" + "=" * 60)
print("FIGURE 4: Ranking preservation by subgroup imbalance")
print("=" * 60)
plot_degradation(
    IMB_VARIANTS, 'rho',
    ylabel='Spearman ρ (mean ± SD)',
    fig_number=4,
    title_text='Treatment-Effect Ranking Preservation Across Privacy Levels by Subgroup Imbalance',
    colors=IMB_COLORS, labels=IMB_LABELS, markers=IMB_MARKERS,
    filename='fig4_rho_by_imbalance.png',
    note='Error bars represent ± 1 SD across 10 Monte Carlo replications.'
)

# =============================================================================
# FIGURE 5: Scatterplot — Spearman ρ vs Qini coefficient (H2)
# Answers: do conditions with better ranking also produce better targeting?
# Each point = one experimental condition (variant × ε combination).
# =============================================================================
print("\n" + "=" * 60)
print("FIGURE 5: Ranking preservation vs downstream performance")
print("=" * 60)

fig, ax = plt.subplots(figsize=(7, 5.5))

# Size-block conditions: open markers
for var in SIZE_VARIANTS:
    df_var = summary[summary['variant'] == var]
    ax.scatter(df_var['rho_mean'], df_var['qini_mean'],
               color=SIZE_COLORS[var], marker=SIZE_MARKERS[var], s=55,
               label=f'Size: {SIZE_LABELS[var]}',
               edgecolors=SIZE_COLORS[var], facecolors='white',
               linewidths=1.5, alpha=0.9, zorder=3)

# Imbalance-block conditions: filled markers
for var in IMB_VARIANTS:
    df_var = summary[summary['variant'] == var]
    ax.scatter(df_var['rho_mean'], df_var['qini_mean'],
               color=IMB_COLORS[var], marker=IMB_MARKERS[var], s=55,
               label=f'Imbalance: {IMB_LABELS[var]}',
               edgecolors=IMB_COLORS[var], facecolors=IMB_COLORS[var],
               linewidths=1.2, alpha=0.7, zorder=3)

# Pearson r across all conditions — used to test H2 (Section 4.5.2)
r_pearson, p_pearson = pearsonr(summary['rho_mean'], summary['qini_mean'])

# Subtle axis reference lines at zero
ax.axhline(y=0, color='black', linestyle='-', alpha=0.15, linewidth=0.6)
ax.axvline(x=0, color='black', linestyle='-', alpha=0.15, linewidth=0.6)

ax.set_xlabel('Mean Spearman ρ')
ax.set_ylabel('Mean Qini coefficient')

# APA: statistics annotation in italic, inside plot — not in the title
ax.text(0.95, 0.05, f'r = {r_pearson:.2f}, p < .001',
        transform=ax.transAxes, fontsize=10, ha='right', va='bottom',
        fontstyle='italic')

leg = ax.legend(loc='upper left', fontsize=9)
leg.get_frame().set_linewidth(0)
leg.get_frame().set_facecolor('none')

apa_figure_title(fig, 5,
    'Relationship Between Ranking Preservation and Downstream Uplift-Model Performance',
    note_text='Each point represents one experimental condition (variant × ε combination).')

plt.subplots_adjust(top=0.86, bottom=0.12)
path = os.path.join(FIGURES_DIR, 'fig5_rho_vs_qini_scatter.png')
plt.savefig(path)
plt.show()
print(f"Saved: {path}")

# =============================================================================
# FIGURE 6: Qini coefficient vs ε — Dataset-size block
# Answers: does DP degrade downstream campaign targeting by dataset size?
# =============================================================================
print("\n" + "=" * 60)
print("FIGURE 6: Qini coefficient by dataset size")
print("=" * 60)
plot_degradation(
    SIZE_VARIANTS, 'qini',
    ylabel='Qini coefficient (mean ± SD)',
    fig_number=6,
    title_text='Downstream Uplift-Model Performance Across Privacy Levels by Dataset Size',
    colors=SIZE_COLORS, labels=SIZE_LABELS, markers=SIZE_MARKERS,
    filename='fig6_qini_by_size.png',
    note='Error bars represent ± 1 SD across 10 Monte Carlo replications.'
)

# =============================================================================
# FIGURE 7: Qini coefficient vs ε — Subgroup-imbalance block
# =============================================================================
print("\n" + "=" * 60)
print("FIGURE 7: Qini coefficient by subgroup imbalance")
print("=" * 60)
plot_degradation(
    IMB_VARIANTS, 'qini',
    ylabel='Qini coefficient (mean ± SD)',
    fig_number=7,
    title_text='Downstream Uplift-Model Performance Across Privacy Levels by Subgroup Imbalance',
    colors=IMB_COLORS, labels=IMB_LABELS, markers=IMB_MARKERS,
    filename='fig7_qini_by_imbalance.png',
    note='Error bars represent ± 1 SD across 10 Monte Carlo replications.'
)


# =============================================================================
# TABLE 7: Main results — size_100 (all ε levels)
# Reports ρ, Qini, and AUUC for the full-data condition (Section 4.4)
# =============================================================================
print("\n" + "=" * 60)
print("TABLE 7: Main uplift results (size_100, outcome: visit)")
print("=" * 60)
print()

df_t = summary[summary['variant'] == 'size_100'].sort_values('eps_num')
print(f"{'ε':<8} {'ρ (mean ± SD)':<22} {'Qini (mean ± SD)':<25} {'AUUC (mean ± SD)':<25} {'Degen':>6}")
print("-" * 90)
print(f"{'Oracle':<8} {'1.0000':<22} {'66.52':<25} {'0.0607':<25} {'0/–':>6}")
for _, row in df_t.iterrows():
    eps_label = '∞' if row['eps_num'] >= 1e5 else f"{row['epsilon']}"
    rho_str   = f"{row['rho_mean']:.4f} ± {row['rho_sd']:.4f}"
    qini_str  = f"{row['qini_mean']:.2f} ± {row['qini_sd']:.2f}"
    auuc_str  = f"{row['auuc_mean']:.6f} ± {row['auuc_sd']:.6f}"
    degen_str = f"{int(row['n_degenerate'])}/{int(row['n_reps'])}"
    print(f"{eps_label:<8} {rho_str:<22} {qini_str:<25} {auuc_str:<25} {degen_str:>6}")


# =============================================================================
# TABLE A12: Spearman rho by dataset size across privacy levels (thesis Table A12).
# Shows how N moderates the ε–ρ relationship (H4)
# =============================================================================
print("\n" + "=" * 60)
print("TABLE A12: Spearman rho by dataset size (across privacy levels)")
print("=" * 60)
print()

key_eps = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 1e6]
print(f"{'Variant':<12}", end="")
for e in key_eps:
    label = '∞' if e >= 1e5 else str(e)
    print(f" {'ε=' + label:>12}", end="")
print(f" {'Degen':>8}")
print("-" * 110)

for var in SIZE_VARIANTS:
    df_v        = summary[summary['variant'] == var].sort_values('eps_num')
    total_degen = 0
    print(f"{SIZE_LABELS[var]:<12}", end="")
    for e in key_eps:
        row = df_v[df_v['eps_num'] == e]
        if len(row) > 0:
            print(f" {row['rho_mean'].values[0]:>12.3f}", end="")
            total_degen += int(row['n_degenerate'].values[0])
        else:
            print(f" {'–':>12}", end="")
    print(f" {total_degen:>6}/70")


# =============================================================================
# Subgroup-imbalance results (thesis Table A13, H5 moderator): rho, Qini and
# degenerate-run counts are built and saved once in the appendix-tables cell
# above (appendix_table_a13_subgroup_imbalance_results.csv).
# =============================================================================

# =============================================================================
# Qini by dataset size - data underlying Figure 6 (no separate thesis table).
# =============================================================================
print("\n" + "=" * 60)
print("Qini by dataset size - data underlying Figure 6 (not a numbered thesis table)")
print("=" * 60)
print()

print(f"{'Variant':<12}", end="")
for e in key_eps:
    label = '∞' if e >= 1e5 else str(e)
    print(f" {'ε=' + label:>12}", end="")
print()
print("-" * 100)

for var in SIZE_VARIANTS:
    df_v = summary[summary['variant'] == var].sort_values('eps_num')
    print(f"{SIZE_LABELS[var]:<12}", end="")
    for e in key_eps:
        row = df_v[df_v['eps_num'] == e]
        if len(row) > 0:
            print(f" {row['qini_mean'].values[0]:>12.2f}", end="")
        else:
            print(f" {'–':>12}", end="")
    print()


# =============================================================================
# HYPOTHESIS TESTING STATISTICS (Section 4.5–4.7)
# =============================================================================
print("\n" + "=" * 60)
print("HYPOTHESIS TESTING STATISTICS")
print("=" * 60)

# --- H1: Does ρ decrease monotonically as ε decreases? ---
# Test: Spearman correlation between ε and mean ρ (size_100 condition only)
print("\n--- H1: ε vs ρ correlation (size_100) ---")
df_h1    = summary[summary['variant'] == 'size_100'].sort_values('eps_num')
eps_vals = df_h1['eps_num'].values
rho_vals = df_h1['rho_mean'].values
r_h1, p_h1 = spearmanr(eps_vals, rho_vals)
print(f"Spearman correlation (ε vs mean ρ): r = {r_h1:.4f}, p = {p_h1:.6f}")
print(f"→ {'Supports' if r_h1 > 0 and p_h1 < 0.05 else 'Does not support'} H1")

# --- H2: Does higher ρ predict higher Qini? ---
# Test: Pearson and Spearman r between condition-level mean ρ and mean Qini
print("\n--- H2: ρ vs Qini correlation (condition-level means) ---")
valid = summary.dropna(subset=['rho_mean', 'qini_mean'])
r_h2_pearson,  p_h2_pearson  = pearsonr(valid['rho_mean'],  valid['qini_mean'])
r_h2_spearman, p_h2_spearman = spearmanr(valid['rho_mean'], valid['qini_mean'])
print(f"Pearson  (condition-level means):  r = {r_h2_pearson:.4f}, p = {p_h2_pearson:.2e}")
print(f"Spearman (condition-level means):  r = {r_h2_spearman:.4f}, p = {p_h2_spearman:.2e}")
print(f"N conditions: {len(valid)}")

# Robustness check: same correlation computed at the individual-run level
r_h2_all, p_h2_all = spearmanr(
    detail['spearman_rho'].dropna(),
    detail.loc[detail['spearman_rho'].notna(), 'qini_coef']
)
print(f"Spearman (all {len(detail)} individual runs): r = {r_h2_all:.4f}, p = {p_h2_all:.2e}")
print(f"→ {'Supports' if r_h2_pearson > 0.3 and p_h2_pearson < 0.05 else 'Does not support'} H2")

# Additional: ρ vs AUUC correlation (complementary to Qini)
r_auuc, p_auuc = pearsonr(valid['rho_mean'], valid['auuc_mean'])
print(f"Pearson  (ρ vs AUUC):              r = {r_auuc:.4f}, p = {p_auuc:.2e}")

# thesis Table A10 - Near-noiseless (eps = inf) size-variant performance; full values in results_summary.csv
# --- H4: Does dataset size moderate the ε–ρ relationship? ---
print("\n--- H4: Dataset-size moderation ---")
print("Mean ρ at ε=∞ (near-noiseless MST baseline):")
for var in SIZE_VARIANTS:
    row = summary[(summary['variant'] == var) & (summary['eps_num'] >= 1e5)]
    if len(row) > 0:
        print(f"  {SIZE_LABELS[var]:>5}: ρ = {row['rho_mean'].values[0]:.4f} ± {row['rho_sd'].values[0]:.4f}")

print("\nSpearman correlation (ε vs ρ) within each size variant:")
for var in SIZE_VARIANTS:
    df_v = summary[summary['variant'] == var].sort_values('eps_num')
    r_v, p_v = spearmanr(df_v['eps_num'], df_v['rho_mean'])
    print(f"  {SIZE_LABELS[var]:>5}: r = {r_v:.4f}, p = {p_v:.4f}")

# --- H5: Does subgroup imbalance moderate the ε–ρ relationship? ---
print("\n--- H5: Subgroup-imbalance moderation ---")
print("Mean ρ at ε=1.0:")
for var in IMB_VARIANTS:
    row = summary[(summary['variant'] == var) & (summary['eps_num'] == 1.0)]
    if len(row) > 0:
        print(f"  {IMB_LABELS[var]:>20}: ρ = {row['rho_mean'].values[0]:.4f} ± {row['rho_sd'].values[0]:.4f}")

print("\nMean ρ at ε=∞:")
for var in IMB_VARIANTS:
    row = summary[(summary['variant'] == var) & (summary['eps_num'] >= 1e5)]
    if len(row) > 0:
        print(f"  {IMB_LABELS[var]:>20}: ρ = {row['rho_mean'].values[0]:.4f} ± {row['rho_sd'].values[0]:.4f}")

print("\nSpearman correlation (ε vs ρ) within each imbalance variant:")
for var in IMB_VARIANTS:
    df_v = summary[summary['variant'] == var].sort_values('eps_num')
    r_v, p_v = spearmanr(df_v['eps_num'], df_v['rho_mean'])
    print(f"  {IMB_LABELS[var]:>20}: r = {r_v:.4f}, p = {p_v:.4f}")


# =============================================================================
# Degeneracy summary (visit, by variant) - supports Table 8 and Tables A6/A7.
# fitted because all synthetic visit labels were the same class (all 0s or all 1s).
# Degenerate runs produce Qini = 0 and ρ = NaN; they are counted but included.
# =============================================================================
print("\n" + "=" * 60)
print("Degeneracy summary (visit, by variant) - see Table 8 and Tables A6/A7")
print("=" * 60)
print()

print(f"{'Variant':<22} {'Visit degen':>12} {'Total runs':>12} {'% degen':>10}")
print("-" * 60)

for var in SIZE_VARIANTS + IMB_VARIANTS:
    df_v        = summary[summary['variant'] == var]
    total_degen = int(df_v['n_degenerate'].sum())
    total_runs  = int(df_v['n_reps'].sum())
    pct         = 100 * total_degen / total_runs if total_runs > 0 else 0
    label       = SIZE_LABELS.get(var, IMB_LABELS.get(var, var))
    print(f"{label:<22} {total_degen:>12} {total_runs:>12} {pct:>9.1f}%")


# =============================================================================
# UTILITY REFERENCE CHECK (descriptive only; not a pre-specified acceptance criterion)
# How close does the best synthetic condition come to the real-data reference Qini?
# Yardstick: 90% of the real-data reference Qini, illustrative only, no fixed cut-off (see Section 3.3.9)
# =============================================================================
print("\n" + "=" * 60)
print("UTILITY REFERENCE CHECK")
print("=" * 60)

ORACLE_QINI = 66.52          # oracle Qini from Phase 3 (real training data)
threshold   = 0.9 * ORACLE_QINI   # 90% of reference Qini, illustrative yardstick only (not a cut-off)
print(f"Oracle Qini: {ORACLE_QINI:.2f}")
print(f"90%-of-reference yardstick: {threshold:.2f}")
print()

print(f"Conditions reaching the 90%-of-reference yardstick (Qini_synthetic ≥ {threshold:.2f}):")
meets = summary[summary['qini_mean'] >= threshold]
if len(meets) == 0:
    print("  NONE — no synthetic condition reaches 90% of the real-data reference Qini")
else:
    for _, row in meets.iterrows():
        eps_label = '∞' if row['eps_num'] >= 1e5 else f"{row['epsilon']}"
        print(f"  {row['variant']} at ε={eps_label}: Qini = {row['qini_mean']:.2f}")

print(f"\nBest synthetic Qini overall: ", end="")
best      = summary.loc[summary['qini_mean'].idxmax()]
eps_label = '∞' if best['eps_num'] >= 1e5 else f"{best['epsilon']}"
print(f"{best['variant']} at ε={eps_label}: Qini = {best['qini_mean']:.2f} "
      f"({100 * best['qini_mean'] / ORACLE_QINI:.1f}% of oracle)")


# =============================================================================
# DONE
# =============================================================================
print("\n" + "=" * 60)
print(f"ALL FIGURES SAVED TO: {FIGURES_DIR}")
print("=" * 60)
print("\nFigures produced:")
for f in sorted(os.listdir(FIGURES_DIR)):
    if f.endswith('.png'):
        print(f"  {f}")
print("\nCopy the printed tables above into your thesis.")
print("Figures and printed tables are numbered to match the thesis.")


Matplotlib is building the font cache; this may take a moment.


FIGURE 3: Ranking preservation by dataset size
Saved: /Users/tumtummer/Downloads/dp-uplift-modelling/results/main/figures/fig3_rho_by_size.png

FIGURE 4: Ranking preservation by subgroup imbalance
Saved: /Users/tumtummer/Downloads/dp-uplift-modelling/results/main/figures/fig4_rho_by_imbalance.png

FIGURE 5: Ranking preservation vs downstream performance
Saved: /Users/tumtummer/Downloads/dp-uplift-modelling/results/main/figures/fig5_rho_vs_qini_scatter.png

FIGURE 6: Qini coefficient by dataset size
Saved: /Users/tumtummer/Downloads/dp-uplift-modelling/results/main/figures/fig6_qini_by_size.png

FIGURE 7: Qini coefficient by subgroup imbalance
Saved: /Users/tumtummer/Downloads/dp-uplift-modelling/results/main/figures/fig7_qini_by_imbalance.png

TABLE 7: Main uplift results (size_100, outcome: visit)

ε        ρ (mean ± SD)          Qini (mean ± SD)          AUUC (mean ± SD)           Degen
------------------------------------------------------------------------------------------
Oracle 

## §1.8 — Outcome-class sparsity diagnostic (conversion vs visit)

Regenerates `conversion_sparsity.csv`: for every synthetic dataset, the per-arm conversion and visit counts and the degeneracy flags that underpin the outcome-event sparsity finding (the conversion arm is degenerate in 489/490 runs, versus 33/490 for visit). Reconstructed from the synthetic datasets in `SYNTH_DIR`, one row per file. Adds an output only; it does not alter any earlier result.

In [12]:
# =============================================================================
# §1.8 — Conversion / visit outcome-class sparsity diagnostic
# Underpins thesis Table 8 (degenerate-run summary by outcome class) and
# Tables A6 (conversion) / A7 (visit) per-condition degenerate-run counts.
# Regenerates conversion_sparsity.csv: per synthetic dataset, the per-arm
# conversion and visit counts plus degeneracy flags underpinning the outcome-
# event sparsity finding (conversion arm degenerate in 489/490 runs).
# =============================================================================
import re
from pathlib import Path
import numpy as np, pandas as pd

_synth = Path(str(SYNTH_DIR))
_out   = Path(str(RESULTS_DIR)) / "conversion_sparsity.csv"

_rows = []
for _f in sorted(_synth.glob("*_eps_*_rep*.csv")):
    _m = re.match(r"(.+)_eps_(.+)_rep(\d+)\.csv$", _f.name)
    if not _m:
        continue
    _variant, _eps, _rep = _m.group(1), _m.group(2), int(_m.group(3))
    _df  = pd.read_csv(_f)
    _trt = _df[_df["treatment"] == 1]
    _ctl = _df[_df["treatment"] == 0]
    _ct, _cc = int(_trt["conversion"].sum()), int(_ctl["conversion"].sum())
    _vt, _vc = int(_trt["visit"].sum()),      int(_ctl["visit"].sum())
    _rows.append({
        "variant": _variant,
        "epsilon": np.inf if _eps == "inf" else float(_eps),
        "rep": _rep,
        "n_total": len(_df),
        "conv_total": int(_df["conversion"].sum()),
        "conv_treated": _ct, "conv_control": _cc,
        "visit_total": int(_df["visit"].sum()),
        "visit_treated": _vt, "visit_control": _vc,
        "conv_degenerate":  (_ct == 0) or (_cc == 0),
        "visit_degenerate": (_vt == 0) or (_vc == 0),
    })

conversion_sparsity = (pd.DataFrame(_rows)
                       .sort_values(["variant", "epsilon", "rep"])
                       .reset_index(drop=True))
conversion_sparsity.to_csv(_out, index=False)

_n  = len(conversion_sparsity)
_cd = int(conversion_sparsity["conv_degenerate"].sum())
_vd = int(conversion_sparsity["visit_degenerate"].sum())
print(f"[ok] wrote {_out}  ({_n} rows)")
print(f"     conversion-arm degenerate: {_cd}/{_n} ({100*_cd/_n:.1f}%)")
print(f"     visit-arm degenerate:      {_vd}/{_n} ({100*_vd/_n:.1f}%)")


[ok] wrote /Users/tumtummer/Downloads/dp-uplift-modelling/results/main/raw_outputs/conversion_sparsity.csv  (490 rows)
     conversion-arm degenerate: 489/490 (99.8%)
     visit-arm degenerate:      33/490 (6.7%)


## §1.9 — Marginal preservation (Table 6 and Table A5)

For every synthetic dataset the column means of treatment, conversion and visit are summarised as mean ± SD across the 10 repetitions per (variant, ε). Table 6 reports the size_100 variant against the original training data; Table A5 reports conversion and visit for every benchmark variant.

In [23]:
# =============================================================================
# §1.9 — Marginal preservation (thesis Table 6 and Table A5)
# =============================================================================
# Synthetic-data quality check: for each synthetic dataset, take the column mean
# of treatment, conversion and visit (the synthetic marginal), then report
# mean ± SD across the 10 repetitions per (variant, ε). Table 6 covers size_100
# against the original training data; Table A5 covers conversion and visit for
# every benchmark variant. SD is the sample standard deviation (ddof = 1).
import glob

_VAR_ORDER = ["size_25", "size_50", "size_75", "size_100",
              "imbalance_baseline", "imbalance_20", "imbalance_05"]
_EPS_ORDER = ["0.1", "0.5", "1.0", "2.0", "5.0", "10.0", "inf"]

_marg_rows = []
for _f in sorted(glob.glob(SYNTH_DIR + "*_eps_*_rep*.csv")):
    _var, _rest = os.path.basename(_f).split("_eps_")
    _eps = _rest.split("_rep")[0]
    _d = pd.read_csv(_f)
    _marg_rows.append({"variant": _var, "epsilon": _eps, "N": len(_d),
                       "treatment": _d["treatment"].mean(),
                       "conversion": _d["conversion"].mean(),
                       "visit": _d["visit"].mean()})

_marg = pd.DataFrame(_marg_rows)
_marg["variant"] = pd.Categorical(_marg["variant"], categories=_VAR_ORDER, ordered=True)
_marg["epsilon"] = pd.Categorical(_marg["epsilon"], categories=_EPS_ORDER, ordered=True)

def _summarise(frame, cols):
    out = frame.groupby(["variant", "epsilon"], observed=True)[cols].agg(["mean", "std"])
    out.columns = [f"{a}_{b}" for a, b in out.columns]
    return out.reset_index().sort_values(["variant", "epsilon"])

# -- Table 6 -- size_100 against the original (real) training marginals --------
_orig = pd.read_csv(DATA_DIR + "size_100.csv")
_t6 = _summarise(_marg[_marg["variant"] == "size_100"], ["N", "treatment", "conversion", "visit"])
print("=" * 74)
print("TABLE 6: Marginal preservation across privacy levels (size_100, N = 44,800)")
print("=" * 74)
print(f"{'ε':>8} {'N(mean)':>9}  {'treatment':>15} {'conversion':>18} {'visit':>15}")
print(f"{'Original':>8} {len(_orig):>9,}  {_orig['treatment'].mean():>15.4f} "
      f"{_orig['conversion'].mean():>18.5f} {_orig['visit'].mean():>15.4f}")
for _, r in _t6.iterrows():
    print(f"{str(r['epsilon']):>8} {r['N_mean']:>9,.0f}  "
          f"{r['treatment_mean']:>7.4f}±{r['treatment_std']:.4f} "
          f"{r['conversion_mean']:>8.5f}±{r['conversion_std']:.5f} "
          f"{r['visit_mean']:>7.4f}±{r['visit_std']:.4f}")

# -- Table A5 -- every variant except imbalance_baseline (≡ size_100 by design)
_a5 = _summarise(_marg[_marg["variant"] != "imbalance_baseline"], ["N", "conversion", "visit"])
print("\n" + "=" * 74)
print("TABLE A5: Marginal preservation across benchmark variants and privacy levels")
print("=" * 74)
print(f"{'variant':>14} {'ε':>6} {'N(mean)':>9}  {'conversion':>18} {'visit':>15}")
for _, r in _a5.iterrows():
    print(f"{r['variant']:>14} {str(r['epsilon']):>6} {r['N_mean']:>9,.0f}  "
          f"{r['conversion_mean']:>8.5f}±{r['conversion_std']:.5f} "
          f"{r['visit_mean']:>7.4f}±{r['visit_std']:.4f}")

# -- Save both tables to the standard raw-outputs folder ----------------------
_t6.round(6).to_csv(RESULTS_DIR + "table6_marginal_preservation_size100.csv", index=False)
_a5.round(6).to_csv(RESULTS_DIR + "appendix_table_a5_marginal_preservation.csv", index=False)
print("\nSaved: table6_marginal_preservation_size100.csv | appendix_table_a5_marginal_preservation.csv")

TABLE 6: Marginal preservation across privacy levels (size_100, N = 44,800)
       ε   N(mean)        treatment         conversion           visit
Original    44,800           0.6671            0.00902          0.1470
     0.1    44,774   0.6663±0.0031  0.00982±0.00269  0.1459±0.0024
     0.5    44,794   0.6669±0.0006  0.00885±0.00041  0.1469±0.0004
     1.0    44,797   0.6670±0.0004  0.00887±0.00027  0.1469±0.0001
     2.0    44,797   0.6670±0.0002  0.00892±0.00015  0.1469±0.0001
     5.0    44,798   0.6671±0.0001  0.00898±0.00006  0.1470±0.0000
    10.0    44,799   0.6671±0.0000  0.00900±0.00003  0.1470±0.0000
     inf    44,799   0.6671±0.0000  0.00901±0.00002  0.1470±0.0000

TABLE A5: Marginal preservation across benchmark variants and privacy levels
       variant      ε   N(mean)          conversion           visit
       size_25    0.1    11,170   0.01808±0.01271  0.1457±0.0124
       size_25    0.5    11,193   0.00910±0.00202  0.1466±0.0022
       size_25    1.0    11,196   0.0

# §2 — Extension: pooling robustness

The robustness of the main findings is examined by splitting the data into separate
treatment-versus-control benchmarks (the men's and women's campaigns), repeating the synthesis and
uplift evaluation on each split, and aggregating the results into the appendix table.


# 1 Helper functions

In [13]:
# ============================================================
# Helper functions: feature encoding, DP synthesis, two-model uplift, and Qini metrics.
# ============================================================
import sys                                                                       # for sys.path injection
import os                                                                        # filesystem
import numpy as np                                                               # numerical arrays
import pandas as pd                                                              # dataframes
from sklearn.linear_model import LogisticRegression                              # Two-Model uplift base learner

# --- private-pgm import (MST lives here) ---
# UPDATE THIS PATH if your private-pgm clone is somewhere else
MECHANISMS_PATH = str(PRIVATE_PGM_MECHANISMS)            # folder containing mst.py
sys.path.insert(0, MECHANISMS_PATH)                                              # make MST importable
from mbi import Dataset, Domain                                                   # MBI graphical-model containers
from mst import MST as run_mst                                                    # the MST synthesiser

# numpy 2.0 renamed np.trapz → np.trapezoid; pick whichever exists
_trapz = getattr(np, 'trapezoid', np.trapz)                                       # version-safe integral

# --- Constants ---
MST_COLS    = ['recency', 'history_segment', 'mens', 'womens', 'zip_code',        # columns passed to MST
               'newbie', 'channel', 'treatment', 'conversion', 'visit', 'spend_bin']
SPEND_BINS  = [-0.01, 0.01, 50, 100, 200, 500]                                    # spend discretisation edges
CAT_COLS    = ['history_segment', 'zip_code', 'channel']                          # MST-side categoricals
DELTA       = 1e-5                                                                # fixed δ for (ε,δ)-DP
FEATURES     = ['recency', 'history_segment', 'mens', 'womens',                   # Two-Model predictors
                'zip_code', 'newbie', 'channel']
CAT_FEATURES = ['history_segment', 'zip_code', 'channel']                         # one-hot these


def prepare_for_mst(df):
    """Encode dataframe into 0-indexed integers required by MST."""
    df = df.copy()                                                                # don't mutate input
    encoding_maps = {}                                                            # store reverse maps
    for col in CAT_COLS:                                                          # label-encode categoricals
        cats = sorted(df[col].unique())                                           # stable category order
        fwd  = {v: i for i, v in enumerate(cats)}                                 # string → int
        bwd  = {i: v for v, i in fwd.items()}                                     # int → string (for decode)
        df[col] = df[col].map(fwd)                                                # apply forward map
        encoding_maps[col] = bwd                                                  # save reverse map
    df['spend_bin'] = pd.cut(df['spend'], bins=SPEND_BINS,                        # discretise spend → 5 bins
                              labels=range(5)).astype(int)
    df['recency']   = df['recency'] - 1                                           # shift recency to 0-indexed
    return df, encoding_maps                                                      # encoded df + maps


def decode_synthetic(synth_df, encoding_maps, original_df):
    """Reverse the encoding so synthetic output matches the benchmark schema."""
    df = synth_df.copy()                                                          # don't mutate input
    for col, bwd in encoding_maps.items():                                        # invert label encoding
        df[col] = df[col].map(bwd)
    df['recency'] = df['recency'] + 1                                             # restore 1-indexed recency
    original = original_df.copy()                                                 # build per-bin spend median
    original['spend_bin'] = pd.cut(original['spend'], bins=SPEND_BINS,
                                    labels=range(5)).astype(int)
    spend_map = {}
    for b in range(5):                                                            # use observed median per bin
        bin_data = original.loc[original['spend_bin'] == b, 'spend']
        spend_map[b] = (bin_data.median() if len(bin_data) > 0
                        else [0, 25, 75, 150, 350][b])                            # fallback midpoints
    df['spend'] = df['spend_bin'].map(spend_map)                                  # decode spend_bin → numeric
    df.loc[df['conversion'] == 0, 'spend'] = 0.0                                  # data rule: no conversion → no spend
    return df[['recency', 'history_segment', 'mens', 'womens', 'zip_code',        # original column order
               'newbie', 'channel', 'treatment', 'conversion', 'visit', 'spend']]


def generate_dp_synthetic(df, epsilon, delta, seed, is_no_dp=False):
    """Generate one synthetic dataset using MST at (ε, δ)."""
    df_enc, encoding_maps = prepare_for_mst(df)                                   # encode for MST
    domain_dict = {col: int(df_enc[col].max()) + 1 for col in MST_COLS}           # domain = max+1 per col
    domain      = Domain.fromdict(domain_dict)                                    # MBI Domain object
    data        = Dataset(df_enc[MST_COLS].values.astype(int), domain)            # wrap as MBI Dataset
    np.random.seed(seed)                                                          # deterministic per rep
    synth_data  = run_mst(data, epsilon=epsilon, delta=delta)                     # run MST
    synth_dict  = synth_data.to_dict()                                            # MBI → dict
    synth_df    = pd.DataFrame(synth_dict, columns=MST_COLS)                      # → dataframe
    return decode_synthetic(synth_df, encoding_maps, df)                          # decode back to schema


def generate_no_dp_synthetic(df, seed):
    """ε = ∞ baseline: MST with negligible noise (ε = 1e6)."""
    return generate_dp_synthetic(df, epsilon=1e6, delta=DELTA,                    # very large ε ≈ no noise
                                  seed=seed, is_no_dp=True)


def encode_features(df, features=FEATURES, cat_features=CAT_FEATURES):
    """One-hot encode predictors for the Two-Model uplift estimator."""
    df_enc = df[features].copy()                                                  # keep only predictors
    df_enc = pd.get_dummies(df_enc, columns=cat_features, drop_first=True)        # one-hot (drop first dummy)
    for col in df_enc.columns:                                                    # cast all to float
        df_enc[col] = df_enc[col].astype(float)
    return df_enc


def _fit_or_constant(X_train, y_train, X_test, label=""):
    """Fit logistic regression; if y_train is single-class, return the constant rate."""
    n_classes = len(np.unique(y_train))                                           # number of distinct labels
    if n_classes < 2:                                                             # degenerate: cannot fit
        return np.full(X_test.shape[0], y_train.mean())                           # fall back to constant
    model = LogisticRegression(max_iter=1000, solver='lbfgs',                     # standard logistic
                                random_state=42, C=1.0)
    model.fit(X_train, y_train)                                                   # fit
    return model.predict_proba(X_test)[:, 1]                                      # P(class=1)


def train_two_model_uplift(df_train, X_test_encoded, test_columns, outcome='visit'):
    """Two-Model uplift: P̂_T(visit|X) − P̂_C(visit|X)."""
    X_train = encode_features(df_train)                                           # encode synthetic predictors
    for col in test_columns:                                                      # align dummy columns to test
        if col not in X_train.columns:
            X_train[col] = 0.0                                                    # missing dummy → 0
    X_train = X_train[test_columns]                                               # enforce identical column order

    treated_mask = df_train['treatment'] == 1                                     # treated arm mask
    control_mask = df_train['treatment'] == 0                                     # control arm mask

    X_treated, y_treated = X_train[treated_mask], df_train.loc[treated_mask, outcome].values  # treated data
    X_control, y_control = X_train[control_mask], df_train.loc[control_mask, outcome].values  # control data

    is_degenerate = ((len(np.unique(y_treated)) < 2) or                           # flag degenerate runs
                     (len(np.unique(y_control)) < 2))

    p_treated = _fit_or_constant(X_treated, y_treated, X_test_encoded, "treated") # P̂_T
    p_control = _fit_or_constant(X_control, y_control, X_test_encoded, "control") # P̂_C
    return p_treated - p_control, is_degenerate                                   # τ̂ + degenerate flag


def compute_qini_curve(uplift_scores, treatment, outcome, n_points=100):
    """Qini curve: per-fraction net uplift in descending uplift order."""
    n                = len(uplift_scores)
    order            = np.argsort(-uplift_scores)                                 # descending sort
    treatment_sorted = treatment[order]
    outcome_sorted   = outcome[order]
    fractions        = np.linspace(0, 1, n_points + 1)                            # x-axis fractions
    qini_values      = np.zeros(n_points + 1)
    n_treated_total  = treatment.sum()
    n_control_total  = n - n_treated_total
    for i, frac in enumerate(fractions):                                          # walk along curve
        if frac == 0:
            continue
        k     = min(int(np.ceil(frac * n)), n)                                    # top-k targeted
        t_top = treatment_sorted[:k]
        y_top = outcome_sorted[:k]
        if n_treated_total > 0 and n_control_total > 0:                           # avoid div-by-zero
            qini_values[i] = ((y_top * t_top).sum() / n_treated_total -
                              (y_top * (1 - t_top)).sum() / n_control_total)
    qini_values = qini_values * n                                                 # scale rates → counts
    return fractions, qini_values


def compute_qini_coefficient(fractions, qini_values):
    """Qini coefficient: AUC of Qini curve minus the random-targeting baseline."""
    auc_qini   = _trapz(qini_values, fractions)                                   # area under Qini curve
    auc_random = 0.5 * qini_values[-1]                                            # area under diagonal
    return auc_qini - auc_random

# 2 Setup

In [14]:
import os                                                                       # filesystem operations
import numpy as np                                                              # numerical arrays and np.inf
import pandas as pd                                                             # dataframes and CSV I/O
from pathlib import Path                                                        # path handling
from scipy.stats import spearmanr                                               # Spearman ranking metric
from sklearn.model_selection import train_test_split                            # stratified train/test split

# --- Paths ---
RAW_CSV            = PROJECT_ROOT / "data" / "raw" / "Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv"  # original Hillstrom file
SPLIT_DATA_DIR     = PROJECT_ROOT / "data" / "extra_analysis" / "pooling_robustness"             # new folder for split benchmarks
SPLIT_SYNTH_DIR    = SPLIT_DATA_DIR / "synthetic_mst"                                                # MST synthetic outputs (optional; we run in-memory)
SPLIT_RESULTS_DIR  = PROJECT_ROOT / "results" / "extra_analysis" / "pooling_robustness" / "raw_outputs"  # results CSVs
for d in [SPLIT_DATA_DIR, SPLIT_SYNTH_DIR, SPLIT_RESULTS_DIR]:                  # ensure all output folders exist
    d.mkdir(parents=True, exist_ok=True)                                        # create if missing (no error if already there)

# --- Settings ---
BASE_SEED = 42                                                                  # same seed as the main pipeline → identical train/test split
EPSILONS  = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0, np.inf]                             # main-analysis ε grid
N_REPS    = 10                                                                  # repetitions per ε per arm

# The helper functions above (encode_features, generate_dp_synthetic, train_two_model_uplift,
# compute_qini_curve, compute_qini_coefficient) are reused for the split-benchmark analysis.

# 3 Two split benchmarks

In [15]:
raw = pd.read_csv(RAW_CSV)                                                      # load raw Hillstrom (~64,000 rows)
print(f"Raw N = {len(raw):,}")                                                  # sanity check overall size
print(raw['segment'].value_counts())                                            # confirm three arms ~21,387 each

# Reproduce the main pipeline's 70/30 train/test split exactly (stratify on segment × visit)
strat_key = raw['segment'].astype(str) + "_" + raw['visit'].astype(str)         # combined stratification key
raw_train, raw_test = train_test_split(                                         # stratified split
    raw, test_size=0.30, random_state=BASE_SEED, stratify=strat_key             # same seed/proportion as the main pipeline
)
print(f"Train N = {len(raw_train):,}  Test N = {len(raw_test):,}")              # expect ~44,800 / ~19,200

def build_split(df, target_segment, label):                                     # helper: build one arm split
    keep = df[df['segment'].isin([target_segment, 'No E-Mail'])].copy()         # keep target arm + control only
    keep['treatment'] = (keep['segment'] == target_segment).astype(int)         # recode binary: 1=target arm, 0=control
    keep = keep.drop(columns=['segment'])                                       # drop the original arm label
    print(f"{label}: N = {len(keep):,}, treat ratio = {keep['treatment'].mean():.3f}")  # confirm ~50/50
    return keep                                                                 # return cleaned dataframe

# --- Men's vs control ---
mens_train = build_split(raw_train, 'Mens E-Mail',   'mens train')              # split train on men's arm
mens_test  = build_split(raw_test,  'Mens E-Mail',   'mens test')               # split test on men's arm
mens_train.to_csv(SPLIT_DATA_DIR / "mens_vs_control_train.csv", index=False)    # persist for replication
mens_test.to_csv (SPLIT_DATA_DIR / "mens_vs_control_test.csv",  index=False)    # persist for replication

# --- Women's vs control ---
womens_train = build_split(raw_train, 'Womens E-Mail', 'womens train')          # symmetric construction
womens_test  = build_split(raw_test,  'Womens E-Mail', 'womens test')           # symmetric construction
womens_train.to_csv(SPLIT_DATA_DIR / "womens_vs_control_train.csv", index=False)
womens_test.to_csv (SPLIT_DATA_DIR / "womens_vs_control_test.csv",  index=False)

Raw N = 64,000
segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64
Train N = 44,800  Test N = 19,200
mens train: N = 29,829, treat ratio = 0.500
mens test: N = 12,784, treat ratio = 0.500
womens train: N = 29,885, treat ratio = 0.501
womens test: N = 12,808, treat ratio = 0.501


# 4 Reference rankings + the synthesis loop

In [16]:
# ============================================================
# Cell 3 — Reference rankings + DP-MST loop (the long one, ~1–2 h)
# ============================================================
results_rows = []                                                               # one dict per (arm, ε, rep)

for arm_label, prefix in [('mens', 'mens_vs_control'),                          # iterate both arms
                          ('womens', 'womens_vs_control')]:
    print(f"\n=== {arm_label.upper()} arm ===")                                 # progress marker

    # --- Load this arm's split data ---
    train_split  = pd.read_csv(SPLIT_DATA_DIR / f"{prefix}_train.csv")          # real split train
    test_split   = pd.read_csv(SPLIT_DATA_DIR / f"{prefix}_test.csv")           # real split holdout
    X_test_split = encode_features(test_split)                                  # encode predictors
    test_cols    = X_test_split.columns.tolist()                                # column order for model fit
    treat_arr    = test_split['treatment'].values                               # binary treatment for Qini
    out_arr      = test_split['visit'].values                                   # primary outcome for Qini

    # --- Split-specific reference (real data, no DP, no MST) ---
    ref_scores, _ = train_two_model_uplift(                                     # Two-Model on real split
        train_split, X_test_split, test_cols, outcome='visit'
    )                                                                           # ref_scores: within-split anchor

    # --- MST + uplift loop: 7 ε × 10 reps = 70 runs per arm ---
    for eps in EPSILONS:                                                        # iterate privacy budgets
        for rep in range(N_REPS):                                               # iterate Monte Carlo reps
            seed_rep = BASE_SEED + rep                                          # vary seed per rep
            if np.isinf(eps):                                                   # ε = ∞ branch
                synth = generate_no_dp_synthetic(train_split, seed=seed_rep)    # near-noiseless MST
            else:                                                               # standard DP-MST
                synth = generate_dp_synthetic(                                  # DP-MST synthesis
                    train_split, epsilon=float(eps),
                    delta=DELTA, seed=seed_rep
                )
            scores, is_degen = train_two_model_uplift(                          # train on synthetic
                synth, X_test_split, test_cols, outcome='visit'
            )
            rho, _        = spearmanr(ref_scores, scores)                       # Spearman ρ vs split reference
            frac, q_curve = compute_qini_curve(                                 # Qini curve on holdout
                scores, treat_arr, out_arr
            )
            qini = compute_qini_coefficient(frac, q_curve)                      # scalar Qini coefficient
            results_rows.append({                                               # log this run
                'arm':           arm_label,
                'epsilon':       eps,
                'rep':           rep,
                'spearman_rho':  rho,
                'qini_coef':     qini,
                'is_degenerate': int(is_degen),
            })
            print(f"  ε={eps}, rep={rep}: ρ={rho:+.4f}, "                       # progress per run
                  f"Qini={qini:+.2f}, degen={is_degen}")

results_df = pd.DataFrame(results_rows)                                         # collect into dataframe
results_df.to_csv(                                                              # persist raw per-rep results
    SPLIT_RESULTS_DIR / "results_split_treatment.csv", index=False
)
print(f"\nSaved {len(results_df)} run rows to results_split_treatment.csv")     # confirmation


=== MENS arm ===
  ε=0.1, rep=0: ρ=+0.1716, Qini=+1.39, degen=False
  ε=0.1, rep=1: ρ=-0.0136, Qini=+7.99, degen=False
  ε=0.1, rep=2: ρ=+0.1352, Qini=+1.47, degen=False
  ε=0.1, rep=3: ρ=+0.0614, Qini=-31.07, degen=False
  ε=0.1, rep=4: ρ=+0.2122, Qini=-1.19, degen=False
  ε=0.1, rep=5: ρ=+0.0003, Qini=-14.01, degen=False
  ε=0.1, rep=6: ρ=+0.2319, Qini=+77.41, degen=True
  ε=0.1, rep=7: ρ=-0.0297, Qini=+56.19, degen=True
  ε=0.1, rep=8: ρ=+0.0858, Qini=+42.69, degen=True
  ε=0.1, rep=9: ρ=+0.1560, Qini=+4.33, degen=False
  ε=0.5, rep=0: ρ=+0.4345, Qini=-8.57, degen=False
  ε=0.5, rep=1: ρ=+0.3840, Qini=+2.13, degen=False
  ε=0.5, rep=2: ρ=+0.3594, Qini=-22.71, degen=False
  ε=0.5, rep=3: ρ=+0.0431, Qini=+3.31, degen=False
  ε=0.5, rep=4: ρ=+0.0395, Qini=+13.93, degen=False
  ε=0.5, rep=5: ρ=+0.0007, Qini=+2.89, degen=False
  ε=0.5, rep=6: ρ=+0.3207, Qini=-39.49, degen=False
  ε=0.5, rep=7: ρ=+0.0408, Qini=-19.87, degen=False
  ε=0.5, rep=8: ρ=+0.3799, Qini=-21.55, degen=False
  ε=0.

# 5 Aggregate to the appendix table

In [17]:
agg = (results_df.groupby(['arm', 'epsilon'], dropna=False)                     # group on (arm, ε)
        .agg(
            rho_mean  = ('spearman_rho',  'mean'),                              # mean ρ across 10 reps
            rho_sd    = ('spearman_rho',  'std'),                               # SD ρ across 10 reps
            qini_mean = ('qini_coef',     'mean'),                              # mean Qini
            qini_sd   = ('qini_coef',     'std'),                               # SD Qini
            n_degen   = ('is_degenerate', 'sum'),                               # number of degenerate runs
            n_reps    = ('rep',           'count'),                             # total reps (should be 10)
        )
        .reset_index())                                                         # flatten back to columns

LABEL = {'mens':   "Men's E-Mail vs control",                                   # display labels
         'womens': "Women's E-Mail vs control"}
agg['Treatment comparison'] = agg['arm'].map(LABEL)                             # human-readable arm
agg['ε']                    = agg['epsilon'].replace(np.inf, '∞').astype(str)   # format ε column
agg['Degenerate runs']      = agg['n_degen'].astype(str) + '/' + agg['n_reps'].astype(str)  # "x/10" format

a14 = agg[['Treatment comparison', 'ε',                                          # final column order
          'rho_mean', 'rho_sd',
          'qini_mean', 'qini_sd',
          'Degenerate runs']].copy()
a14[['rho_mean', 'rho_sd']]   = a14[['rho_mean', 'rho_sd']].round(4)              # round ρ to 4 dp

a14.to_csv(                                                                       # export the appendix-ready table
    SPLIT_RESULTS_DIR / "appendix_table_a14_split_robustness.csv", index=False    # thesis Table A14 — separate e-mail treatment robustness check
)
print(a14.to_string(index=False))                                                 # show the table inline

     Treatment comparison    ε  rho_mean  rho_sd  qini_mean   qini_sd Degenerate runs
  Men's E-Mail vs control  0.1    0.1011  0.0948  14.520000 33.521932            3/10
  Men's E-Mail vs control  0.5    0.2352  0.1784  -9.632000 16.120506            0/10
  Men's E-Mail vs control  1.0    0.2055  0.1672  -6.222000 21.842400            0/10
  Men's E-Mail vs control  2.0    0.1305  0.1228  15.306000  7.938849            0/10
  Men's E-Mail vs control  5.0    0.0653  0.0095  18.008000  2.325099            0/10
  Men's E-Mail vs control 10.0    0.0637  0.0064  17.124000  2.414789            0/10
  Men's E-Mail vs control    ∞    0.0621  0.0074  17.682000  2.869211            0/10
Women's E-Mail vs control  0.1    0.0985  0.1420  54.931638 33.710748            9/10
Women's E-Mail vs control  0.5   -0.1833  0.2373 -39.695885 17.898687            0/10
Women's E-Mail vs control  1.0   -0.2135  0.0154  -1.951897  3.523943            0/10
Women's E-Mail vs control  2.0   -0.1972  0.0156  -0.5

# §3 — Extension: the ε sweet spot

The privacy-budget region around the practical elbow is examined to locate the "sweet spot", where
utility is retained at the strongest feasible privacy. New benchmark variants are constructed, the
synthesis and uplift scoring are re-run, and the results are compared against the main pipeline.


# 1 Helper functions

In [18]:
# ============================================================
# Helper functions: feature encoding, DP synthesis, two-model uplift, and Qini metrics.
# ============================================================
import sys                                                                       # for sys.path injection
import os                                                                        # filesystem
import numpy as np                                                               # numerical arrays
import pandas as pd                                                              # dataframes
from sklearn.linear_model import LogisticRegression                              # Two-Model uplift base learner

# --- private-pgm import (MST lives here) ---
# UPDATE THIS PATH if your private-pgm clone is somewhere else
MECHANISMS_PATH = str(PRIVATE_PGM_MECHANISMS)            # folder containing mst.py
sys.path.insert(0, MECHANISMS_PATH)                                              # make MST importable
from mbi import Dataset, Domain                                                   # MBI graphical-model containers
from mst import MST as run_mst                                                    # the MST synthesiser

# numpy 2.0 renamed np.trapz → np.trapezoid; pick whichever exists
_trapz = getattr(np, 'trapezoid', np.trapz)                                       # version-safe integral

# --- Constants ---
MST_COLS    = ['recency', 'history_segment', 'mens', 'womens', 'zip_code',        # columns passed to MST
               'newbie', 'channel', 'treatment', 'conversion', 'visit', 'spend_bin']
SPEND_BINS  = [-0.01, 0.01, 50, 100, 200, 500]                                    # spend discretisation edges
CAT_COLS    = ['history_segment', 'zip_code', 'channel']                          # MST-side categoricals
DELTA       = 1e-5                                                                # fixed δ for (ε,δ)-DP
FEATURES     = ['recency', 'history_segment', 'mens', 'womens',                   # Two-Model predictors
                'zip_code', 'newbie', 'channel']
CAT_FEATURES = ['history_segment', 'zip_code', 'channel']                         # one-hot these


def prepare_for_mst(df):
    """Encode dataframe into 0-indexed integers required by MST."""
    df = df.copy()                                                                # don't mutate input
    encoding_maps = {}                                                            # store reverse maps
    for col in CAT_COLS:                                                          # label-encode categoricals
        cats = sorted(df[col].unique())                                           # stable category order
        fwd  = {v: i for i, v in enumerate(cats)}                                 # string → int
        bwd  = {i: v for v, i in fwd.items()}                                     # int → string (for decode)
        df[col] = df[col].map(fwd)                                                # apply forward map
        encoding_maps[col] = bwd                                                  # save reverse map
    df['spend_bin'] = pd.cut(df['spend'], bins=SPEND_BINS,                        # discretise spend → 5 bins
                              labels=range(5)).astype(int)
    df['recency']   = df['recency'] - 1                                           # shift recency to 0-indexed
    return df, encoding_maps                                                      # encoded df + maps


def decode_synthetic(synth_df, encoding_maps, original_df):
    """Reverse the encoding so synthetic output matches the benchmark schema."""
    df = synth_df.copy()                                                          # don't mutate input
    for col, bwd in encoding_maps.items():                                        # invert label encoding
        df[col] = df[col].map(bwd)
    df['recency'] = df['recency'] + 1                                             # restore 1-indexed recency
    original = original_df.copy()                                                 # build per-bin spend median
    original['spend_bin'] = pd.cut(original['spend'], bins=SPEND_BINS,
                                    labels=range(5)).astype(int)
    spend_map = {}
    for b in range(5):                                                            # use observed median per bin
        bin_data = original.loc[original['spend_bin'] == b, 'spend']
        spend_map[b] = (bin_data.median() if len(bin_data) > 0
                        else [0, 25, 75, 150, 350][b])                            # fallback midpoints
    df['spend'] = df['spend_bin'].map(spend_map)                                  # decode spend_bin → numeric
    df.loc[df['conversion'] == 0, 'spend'] = 0.0                                  # data rule: no conversion → no spend
    return df[['recency', 'history_segment', 'mens', 'womens', 'zip_code',        # original column order
               'newbie', 'channel', 'treatment', 'conversion', 'visit', 'spend']]


def generate_dp_synthetic(df, epsilon, delta, seed, is_no_dp=False):
    """Generate one synthetic dataset using MST at (ε, δ)."""
    df_enc, encoding_maps = prepare_for_mst(df)                                   # encode for MST
    domain_dict = {col: int(df_enc[col].max()) + 1 for col in MST_COLS}           # domain = max+1 per col
    domain      = Domain.fromdict(domain_dict)                                    # MBI Domain object
    data        = Dataset(df_enc[MST_COLS].values.astype(int), domain)            # wrap as MBI Dataset
    np.random.seed(seed)                                                          # deterministic per rep
    synth_data  = run_mst(data, epsilon=epsilon, delta=delta)                     # run MST
    synth_dict  = synth_data.to_dict()                                            # MBI → dict
    synth_df    = pd.DataFrame(synth_dict, columns=MST_COLS)                      # → dataframe
    return decode_synthetic(synth_df, encoding_maps, df)                          # decode back to schema


def generate_no_dp_synthetic(df, seed):
    """ε = ∞ baseline: MST with negligible noise (ε = 1e6)."""
    return generate_dp_synthetic(df, epsilon=1e6, delta=DELTA,                    # very large ε ≈ no noise
                                  seed=seed, is_no_dp=True)


def encode_features(df, features=FEATURES, cat_features=CAT_FEATURES):
    """One-hot encode predictors for the Two-Model uplift estimator."""
    df_enc = df[features].copy()                                                  # keep only predictors
    df_enc = pd.get_dummies(df_enc, columns=cat_features, drop_first=True)        # one-hot (drop first dummy)
    for col in df_enc.columns:                                                    # cast all to float
        df_enc[col] = df_enc[col].astype(float)
    return df_enc


def _fit_or_constant(X_train, y_train, X_test, label=""):
    """Fit logistic regression; if y_train is single-class, return the constant rate."""
    n_classes = len(np.unique(y_train))                                           # number of distinct labels
    if n_classes < 2:                                                             # degenerate: cannot fit
        return np.full(X_test.shape[0], y_train.mean())                           # fall back to constant
    model = LogisticRegression(max_iter=1000, solver='lbfgs',                     # standard logistic
                                random_state=42, C=1.0)
    model.fit(X_train, y_train)                                                   # fit
    return model.predict_proba(X_test)[:, 1]                                      # P(class=1)


def train_two_model_uplift(df_train, X_test_encoded, test_columns, outcome='visit'):
    """Two-Model uplift: P̂_T(visit|X) − P̂_C(visit|X)."""
    X_train = encode_features(df_train)                                           # encode synthetic predictors
    for col in test_columns:                                                      # align dummy columns to test
        if col not in X_train.columns:
            X_train[col] = 0.0                                                    # missing dummy → 0
    X_train = X_train[test_columns]                                               # enforce identical column order

    treated_mask = df_train['treatment'] == 1                                     # treated arm mask
    control_mask = df_train['treatment'] == 0                                     # control arm mask

    X_treated, y_treated = X_train[treated_mask], df_train.loc[treated_mask, outcome].values  # treated data
    X_control, y_control = X_train[control_mask], df_train.loc[control_mask, outcome].values  # control data

    is_degenerate = ((len(np.unique(y_treated)) < 2) or                           # flag degenerate runs
                     (len(np.unique(y_control)) < 2))

    p_treated = _fit_or_constant(X_treated, y_treated, X_test_encoded, "treated") # P̂_T
    p_control = _fit_or_constant(X_control, y_control, X_test_encoded, "control") # P̂_C
    return p_treated - p_control, is_degenerate                                   # τ̂ + degenerate flag


def compute_qini_curve(uplift_scores, treatment, outcome, n_points=100):
    """Qini curve: per-fraction net uplift in descending uplift order."""
    n                = len(uplift_scores)
    order            = np.argsort(-uplift_scores)                                 # descending sort
    treatment_sorted = treatment[order]
    outcome_sorted   = outcome[order]
    fractions        = np.linspace(0, 1, n_points + 1)                            # x-axis fractions
    qini_values      = np.zeros(n_points + 1)
    n_treated_total  = treatment.sum()
    n_control_total  = n - n_treated_total
    for i, frac in enumerate(fractions):                                          # walk along curve
        if frac == 0:
            continue
        k     = min(int(np.ceil(frac * n)), n)                                    # top-k targeted
        t_top = treatment_sorted[:k]
        y_top = outcome_sorted[:k]
        if n_treated_total > 0 and n_control_total > 0:                           # avoid div-by-zero
            qini_values[i] = ((y_top * t_top).sum() / n_treated_total -
                              (y_top * (1 - t_top)).sum() / n_control_total)
    qini_values = qini_values * n                                                 # scale rates → counts
    return fractions, qini_values


def compute_qini_coefficient(fractions, qini_values):
    """Qini coefficient: AUC of Qini curve minus the random-targeting baseline."""
    auc_qini   = _trapz(qini_values, fractions)                                   # area under Qini curve
    auc_random = 0.5 * qini_values[-1]                                            # area under diagonal
    return auc_qini - auc_random

# 2 Paths + ε grid + helper sanity check

In [19]:
# =============================================================================
# Paths, settings, and helper checks for the sweet-spot extension
# =============================================================================

import os                                                                       # file-path handling
import numpy as np                                                              # numerical operations
import pandas as pd                                                             # dataframes and CSV I/O
import matplotlib.pyplot as plt                                                 # plotting
from pathlib import Path                                                        # clean path objects
from scipy.stats import spearmanr                                               # Spearman ranking preservation
from sklearn.model_selection import train_test_split                            # stratified subsampling

MAIN_DATA = PROJECT_ROOT / "data" / "main_benchmarks"                        # main train/test/benchmark CSVs
MAIN_SUMMARY = PROJECT_ROOT / "results" / "main" / "raw_outputs" / "results_summary.csv"  # main results summary

SWEET_ROOT = PROJECT_ROOT / "data" / "extra_analysis" / "sweetspot_v2"        # versioned sweet-spot data folder
SWEET_DATA = SWEET_ROOT / "benchmark_variants"                                  # new real benchmark variants
SWEET_SYNTH = SWEET_ROOT / "synthetic_mst"                                      # new synthetic MST datasets
SWEET_RES = PROJECT_ROOT / "results" / "extra_analysis" / "sweetspot_v2"     # versioned sweet-spot results folder

for p in [SWEET_DATA, SWEET_SYNTH, SWEET_RES / "raw_outputs", SWEET_RES / "tables", SWEET_RES / "figures"]:
    p.mkdir(parents=True, exist_ok=True)                                        # create folders if missing

BASE_SEED = 42                                                                  # match main pipeline seed
N_REPS = 10                                                                     # match main pipeline repetitions
DELTA = 1e-5                                                                    # match main pipeline DP delta
EPS_STAR = 0.5                                                                  # practical ε-elbow from Item 1.5

RUN_FULL_EPS = True                                                            # set True only for full ε visual curves
EPSILONS = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0, np.inf] if RUN_FULL_EPS else [EPS_STAR]  # controlled vs visual grid

STRAT_OUTCOME = "conversion"                                                    # sampling safeguard; matches main pipeline
PRIMARY_OUTCOME = "visit"                                                       # actual uplift outcome used in thesis

required_helpers = [                                                            # helper functions reused from the main pipeline
    "generate_dp_synthetic",
    "generate_no_dp_synthetic",
    "train_two_model_uplift",
    "compute_qini_curve",
    "compute_qini_coefficient",
    "encode_features",
]

missing = [fn for fn in required_helpers if fn not in globals()]                 # check helper availability
if missing:
    raise RuntimeError(f"Missing helper functions: {missing}")                   # fail early if helper cell was not run

print(f"Sweet-spot extension folder: {SWEET_RES}")                              # confirm output folder
print(f"ε grid: {EPSILONS}")                                                     # confirm ε design

Sweet-spot extension folder: /Users/tumtummer/Downloads/dp-uplift-modelling/results/extra_analysis/sweetspot_v2
ε grid: [0.1, 0.5, 1.0, 2.0, 5.0, 10.0, inf]


# 3 new variants from train.csv, validate, save (builds Table A15)

In [20]:
# =============================================================================
# Cell 2 — Build new sweet-spot benchmark variants
# =============================================================================

df_train = pd.read_csv(MAIN_DATA / "train.csv")                                  # load full training partition

df_train["_strat"] = (                                                           # recreate main-pipeline stratification key
    df_train["treatment"].astype(str) + "_" + df_train[STRAT_OUTCOME].astype(str)
)                                                                                # treatment × conversion, not treatment × visit

def _make_size(df, frac, seed=BASE_SEED):                                        # create one size variant
    out, _ = train_test_split(                                                   # mirror main pipeline subsampling
        df,
        train_size=frac,
        random_state=seed,
        stratify=df["_strat"],
    )
    return out.drop(columns=["_strat"]).reset_index(drop=True)                   # remove helper column before saving

def _make_imbalance(df, target_share, seed=BASE_SEED):                           # create one newbie-share variant
    rng = np.random.RandomState(seed)                                            # deterministic sampler
    clean = df.drop(columns=["_strat"], errors="ignore")                         # remove helper column
    newbies = clean[clean["newbie"] == 1]                                        # pool of new customers
    existing = clean[clean["newbie"] == 0]                                       # pool of existing customers

    n_total = len(clean)                                                         # keep full training size
    n_new = int(n_total * target_share)                                          # target number of newbies
    n_existing = n_total - n_new                                                 # remaining rows are existing customers

    sampled_new = newbies.sample(                                                # sample newbie rows
        n=n_new,
        replace=n_new > len(newbies),
        random_state=rng,
    )
    sampled_existing = existing.sample(                                          # sample existing-customer rows
        n=n_existing,
        replace=n_existing > len(existing),
        random_state=rng,
    )

    out = pd.concat([sampled_new, sampled_existing], ignore_index=True)          # combine sampled groups
    return out.sample(frac=1, random_state=rng).reset_index(drop=True)           # shuffle rows

VARIANTS = {                                                                     # minimal controlled infill variants
    "size_60": _make_size(df_train, 0.60),                                       # infill between size_50 and size_75
    "size_70": _make_size(df_train, 0.70),                                       # infill closer to size_75
    "imbalance_10": _make_imbalance(df_train, 0.10),                             # infill between 5% and 20%
    "imbalance_35": _make_imbalance(df_train, 0.35),                             # infill between 20% and 50%
}

validation_rows = []                                                             # store validation diagnostics

for name, dfv in VARIANTS.items():                                               # save and validate each variant
    dfv.to_csv(SWEET_DATA / f"{name}.csv", index=False)                          # save real benchmark variant

    validation_rows.append({                                                     # collect construction checks
        "variant": name,
        "n": len(dfv),
        "treatment_ratio": dfv["treatment"].mean(),
        "visit_rate": dfv["visit"].mean(),
        "conversion_rate": dfv["conversion"].mean(),
        "newbie_share": dfv["newbie"].mean(),
        "treated_visits": dfv.loc[dfv["treatment"] == 1, "visit"].sum(),
        "control_visits": dfv.loc[dfv["treatment"] == 0, "visit"].sum(),
        "treated_conversions": dfv.loc[dfv["treatment"] == 1, "conversion"].sum(),
        "control_conversions": dfv.loc[dfv["treatment"] == 0, "conversion"].sum(),
    })

validation = pd.DataFrame(validation_rows)                                       # create validation table
validation.to_csv(SWEET_RES / "raw_outputs" / "validation_sweetspot_variants.csv", index=False)  # save checks

# Appendix Table A15 — Exploratory Infill Variant Validation (thesis Table A15).
print("\n" + "=" * 60)
print("TABLE A15: Exploratory Infill Variant Validation")
print("=" * 60)
print(validation.to_string(index=False))                                         # inspect before synthesis


TABLE A15: Exploratory Infill Variant Validation
     variant     n  treatment_ratio  visit_rate  conversion_rate  newbie_share  treated_visits  control_visits  treated_conversions  control_conversions
     size_60 26880         0.667113    0.147210         0.009040      0.498177            3044             913                  192                   51
     size_70 31359         0.667081    0.145764         0.009025      0.500143            3513            1058                  223                   60
imbalance_10 44800         0.667500    0.167879         0.009799      0.100000            5621            1900                  320                  119
imbalance_35 44800         0.665826    0.153795         0.009665      0.349978            5233            1657                  337                   96


# 4 oracle ref + MST + score loop

In [21]:
# =============================================================================
# Cell 3 — Run MST synthesis and evaluate sweet-spot variants
# =============================================================================

df_test = pd.read_csv(MAIN_DATA / "test.csv")                                    # load unchanged holdout test set
X_test = encode_features(df_test)                                                # encode holdout predictors
test_cols = X_test.columns.tolist()                                              # store test feature order
treat_arr = df_test["treatment"].values                                          # treatment indicator for Qini
out_arr = df_test[PRIMARY_OUTCOME].values                                        # primary outcome: visit

oracle_train = pd.read_csv(MAIN_DATA / "size_100.csv")                           # full real-data training benchmark
oracle_scores, _ = train_two_model_uplift(                                       # reference ranking from real data
    oracle_train,
    X_test,
    test_cols,
    outcome=PRIMARY_OUTCOME,
)

def _eps_label(eps):                                                             # clean filename label for ε
    return "inf" if np.isinf(eps) else str(eps).replace(".", "p")

rows = []                                                                        # one row per variant × ε × repetition

for name, train_v in VARIANTS.items():                                           # loop over new variants
    print(f"\n=== {name} ===")                                                   # progress marker

    for eps in EPSILONS:                                                         # loop over ε grid
        for rep in range(N_REPS):                                                # 10 repetitions
            seed = BASE_SEED + rep                                               # match main seed pattern
            synth_path = SWEET_SYNTH / f"{name}_eps_{_eps_label(eps)}_rep{rep}.csv"  # cache path

            if synth_path.exists():                                              # reuse if already generated
                synth = pd.read_csv(synth_path)
            else:
                synth = (                                                        # generate synthetic dataset
                    generate_no_dp_synthetic(train_v, seed=seed)
                    if np.isinf(eps)
                    else generate_dp_synthetic(train_v, epsilon=float(eps), delta=DELTA, seed=seed)
                )
                synth.to_csv(synth_path, index=False)                            # cache synthetic data

            scores, degen = train_two_model_uplift(                              # train on synthetic, score holdout
                synth,
                X_test,
                test_cols,
                outcome=PRIMARY_OUTCOME,
            )

            rho = spearmanr(oracle_scores, scores).correlation if np.std(scores) > 0 else np.nan  # ranking preservation
            frac, q_curve = compute_qini_curve(scores, treat_arr, out_arr)       # Qini curve
            qini = compute_qini_coefficient(frac, q_curve)                       # Qini coefficient

            rows.append({                                                        # store run-level result
                "variant": name,
                "epsilon": eps,
                "rep": rep,
                "spearman_rho": rho,
                "qini_coef": qini,
                "is_degenerate": int(degen),
            })

            print(f"ε={eps}, rep={rep}: ρ={rho:+.3f}, Qini={qini:+.2f}, degen={degen}")  # progress log

runs_df = pd.DataFrame(rows)                                                     # run-level dataframe
runs_df.to_csv(SWEET_RES / "raw_outputs" / "results_sweetspot_runs.csv", index=False)  # save run-level results
# Run-level source for the sweet-spot aggregate and thesis Tables A16 / A17.

print(f"\nSaved {len(runs_df)} run rows.")                                       # confirmation


=== size_60 ===
ε=0.1, rep=0: ρ=+0.400, Qini=-23.91, degen=True
ε=0.1, rep=1: ρ=+0.464, Qini=-13.91, degen=True
ε=0.1, rep=2: ρ=+0.242, Qini=-33.62, degen=True
ε=0.1, rep=3: ρ=+0.173, Qini=+19.48, degen=False
ε=0.1, rep=4: ρ=+0.080, Qini=-5.34, degen=False
ε=0.1, rep=5: ρ=+0.070, Qini=-24.88, degen=True
ε=0.1, rep=6: ρ=+0.527, Qini=+5.60, degen=True
ε=0.1, rep=7: ρ=+0.327, Qini=-29.90, degen=True
ε=0.1, rep=8: ρ=+0.538, Qini=+2.05, degen=True
ε=0.1, rep=9: ρ=+0.415, Qini=-12.92, degen=True
ε=0.5, rep=0: ρ=+0.325, Qini=+36.97, degen=False
ε=0.5, rep=1: ρ=-0.214, Qini=-30.46, degen=False
ε=0.5, rep=2: ρ=+0.238, Qini=+27.57, degen=False
ε=0.5, rep=3: ρ=+0.263, Qini=-1.77, degen=False
ε=0.5, rep=4: ρ=+0.371, Qini=+0.66, degen=False
ε=0.5, rep=5: ρ=+0.382, Qini=+20.85, degen=False
ε=0.5, rep=6: ρ=-0.284, Qini=-22.64, degen=False
ε=0.5, rep=7: ρ=-0.232, Qini=-43.84, degen=False
ε=0.5, rep=8: ρ=-0.249, Qini=-27.61, degen=False
ε=0.5, rep=9: ρ=+0.336, Qini=+18.87, degen=False
ε=1.0, rep=0: ρ=

# 5 aggregate + appendix Tables A16/A17 + drop-in Figures A2–A5

In [22]:
# === Cell 5: aggregate + concat with main + ρ-primary / Qini-companion figures ===
agg = (runs_df.groupby(['variant','epsilon'], dropna=False)                     # group per (variant, ε)
       .agg(rho_mean=('spearman_rho','mean'),  rho_sd=('spearman_rho','std'),   # ρ aggregates
            qini_mean=('qini_coef','mean'),    qini_sd=('qini_coef','std'),    # Qini aggregates
            n_degenerate=('is_degenerate','sum'), n_reps=('rep','count'))
       .reset_index())                                                          # flatten to columns
agg.to_csv(SWEET_RES / "raw_outputs" / "summary_sweetspot.csv", index=False)    # save sweet-spot aggregate

# =============================================================================
# TABLE A16: Full ε-grid results for the size infill variants (thesis Table A16)
# Source: sweet-spot aggregate `agg`; N per variant from the validation table.
# =============================================================================
_val_a16 = pd.read_csv(SWEET_RES / "raw_outputs" / "validation_sweetspot_variants.csv")  # infill N lookup
_N_by_variant = dict(zip(_val_a16["variant"], _val_a16["n"]))                    # variant -> sample size
_a16 = agg[agg["variant"].isin(["size_60", "size_70"])].sort_values(["variant", "epsilon"])  # size infill, full grid
_a16_out = pd.DataFrame({
    "Variant":         _a16["variant"].values,
    "ε":               _a16["epsilon"].replace(np.inf, "∞").astype(str).values,
    "N":               _a16["variant"].map(_N_by_variant).values,
    "Mean ρ":          _a16["rho_mean"].round(4).values,
    "SD ρ":            _a16["rho_sd"].round(4).values,
    "Mean Qini":       _a16["qini_mean"].round(2).values,
    "SD Qini":         _a16["qini_sd"].round(2).values,
    "Degenerate runs": (_a16["n_degenerate"].astype(int).astype(str) + "/" + _a16["n_reps"].astype(int).astype(str)).values,
})
_a16_out.to_csv(SWEET_RES / "raw_outputs" / "appendix_table_a16_size_infill_full_grid.csv", index=False)  # save A16
print("\n" + "=" * 60)
print("TABLE A16: Full ε-grid results for the size infill variants")
print("=" * 60)
print(_a16_out.to_string(index=False))

# =============================================================================
# TABLE A17: Exploratory infill results at ε = 0.5 (thesis Table A17)
# Imbalance infill variants at the conservative elbow ε* = 0.5 (EPS_STAR).
# =============================================================================
_a17 = agg[(agg["variant"].isin(["imbalance_10", "imbalance_35"])) & (np.isclose(agg["epsilon"], EPS_STAR))].sort_values("variant")
_a17_out = pd.DataFrame({
    "Variant":         _a17["variant"].values,
    "Mean ρ":          _a17["rho_mean"].round(4).values,
    "SD ρ":            _a17["rho_sd"].round(4).values,
    "Mean Qini":       _a17["qini_mean"].round(2).values,
    "SD Qini":         _a17["qini_sd"].round(2).values,
    "Degenerate runs": (_a17["n_degenerate"].astype(int).astype(str) + "/" + _a17["n_reps"].astype(int).astype(str)).values,
})
_a17_out.to_csv(SWEET_RES / "raw_outputs" / "appendix_table_a17_imbalance_infill_eps0p5.csv", index=False)  # save A17
print("\n" + "=" * 60)
print("TABLE A17: Exploratory infill results at ε = 0.5")
print("=" * 60)
print(_a17_out.to_string(index=False))


main_agg = pd.read_csv(MAIN_SUMMARY)                                            # load existing main-pipeline aggregates
main_agg['epsilon'] = main_agg['epsilon'].astype(float)                         # 'inf' string → np.inf for sorting
keep = ['variant','epsilon','rho_mean','rho_sd','qini_mean','qini_sd']          # carry BOTH metrics through
combined = pd.concat([main_agg[keep], agg[keep]], ignore_index=True)            # 7 main + 4 new = 11 variants
# =============================================================================
# APA-style plotting block matching main pipeline figures
# =============================================================================

import matplotlib.font_manager as fm                                             # font detection
import subprocess                                                                # optional font-cache check
import matplotlib.pyplot as plt                                                  # plotting

def _setup_times_new_roman():                                                    # match main pipeline font setup
    fm._load_fontmanager(try_read_cache=False)                                   # refresh matplotlib font cache
    available = {f.name for f in fm.fontManager.ttflist}                         # available font names

    if "Times New Roman" in available:                                           # preferred APA thesis font
        return "Times New Roman"

    try:                                                                         # macOS/Linux fallback check
        subprocess.run(["fc-list", ":family=Times New Roman"],
                       capture_output=True, check=True)
        fm._load_fontmanager(try_read_cache=False)
        available = {f.name for f in fm.fontManager.ttflist}
        if "Times New Roman" in available:
            return "Times New Roman"
    except Exception:
        pass

    print("⚠ Times New Roman not found — using STIXGeneral fallback")             # same fallback logic
    return "STIXGeneral"

TNR = _setup_times_new_roman()                                                   # selected serif font

plt.rcParams.update({                                                            # main-pipeline APA style
    "font.family": "serif",
    "font.serif": [TNR, "Times New Roman", "Times", "STIXGeneral", "DejaVu Serif"],
    "font.size": 11,
    "mathtext.fontset": "stix",

    "figure.dpi": 150,
    "figure.facecolor": "white",
    "figure.edgecolor": "white",
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.facecolor": "white",
    "savefig.edgecolor": "white",
    "savefig.pad_inches": 0.15,

    "axes.facecolor": "white",
    "axes.edgecolor": "black",
    "axes.linewidth": 0.8,
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.labelsize": 12,
    "axes.labelweight": "normal",
    "axes.titlesize": 12,
    "axes.titleweight": "normal",

    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.size": 4,
    "ytick.major.size": 4,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.minor.visible": False,
    "ytick.minor.visible": False,

    "legend.fontsize": 10,
    "legend.frameon": False,
    "legend.fancybox": False,
    "legend.shadow": False,

    "lines.linewidth": 1.5,
    "lines.markersize": 6,
})

# --- Epsilon ordering: categorical x-axis, matching main pipeline figures ---
EPS_ORDER = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0, np.inf]                              # canonical ε order
EPS_LABELS = ["0.1", "0.5", "1", "2", "5", "10", "∞"]                             # display labels
combined["epsilon"] = combined["epsilon"].apply(                                  # robust ε parser
    lambda x: np.inf if str(x).lower() in {"inf", "infinity", "∞"} else float(x)
)

# --- Colours: preserve main-pipeline colours, add new infill colours ---
SIZE_COLORS = {
    "size_25":  "#D62728",                                                        # same as main: 25%
    "size_50":  "#FF7F0E",                                                        # same as main: 50%
    "size_60":  "#9467BD",                                                        # new infill: 60%
    "size_70":  "#8C564B",                                                        # new infill: 70%
    "size_75":  "#2CA02C",                                                        # same as main: 75%
    "size_100": "#1F77B4",                                                        # same as main: 100%
}

SIZE_MARKERS = {
    "size_25": "v",                                                               # same as main
    "size_50": "s",                                                               # same as main
    "size_60": "D",                                                               # new infill marker
    "size_70": "P",                                                               # new infill marker
    "size_75": "^",                                                               # same as main
    "size_100": "o",                                                              # same as main
}

IMB_COLORS = {
    "imbalance_05":       "#D62728",                                              # same as main: 5%
    "imbalance_10":       "#9467BD",                                              # new infill: 10%
    "imbalance_20":       "#FF7F0E",                                              # same as main: 20%
    "imbalance_35":       "#2CA02C",                                              # new infill: 35%
    "imbalance_baseline": "#1F77B4",                                              # same as main: baseline
}

IMB_MARKERS = {
    "imbalance_05":       "v",                                                    # same as main
    "imbalance_10":       "D",                                                    # new infill marker
    "imbalance_20":       "s",                                                    # same as main
    "imbalance_35":       "^",                                                    # new infill marker
    "imbalance_baseline": "o",                                                    # same as main
}

def apa_figure_title(fig, fig_number, title_text, note_text=None):                # APA figure header
    fig.text(0.02, 1.04, f"Figure {fig_number}",
             fontsize=12, fontweight="bold", fontstyle="normal",
             fontfamily="serif", va="bottom", transform=fig.transFigure)

    fig.text(0.02, 0.99, title_text,
             fontsize=12, fontweight="normal", fontstyle="italic",
             fontfamily="serif", va="bottom", transform=fig.transFigure)

    if note_text:
        fig.text(0.02, -0.04, f"Note. {note_text}",
                 fontsize=10, fontweight="normal", fontstyle="italic",
                 fontfamily="serif", va="top", transform=fig.transFigure)

def _series_for_variant(df, variant, metric):                                     # get y/yerr in fixed ε order
    sub = df[df["variant"] == variant].copy()                                     # filter one variant
    y_vals, y_errs = [], []                                                       # containers

    for eps in EPS_ORDER:                                                         # enforce main-pipeline ε order
        row = sub[np.isinf(sub["epsilon"])] if np.isinf(eps) else sub[np.isclose(sub["epsilon"], eps)]
        if len(row) > 0:
            y_vals.append(row[f"{metric}_mean"].values[0])                       # mean value
            y_errs.append(row[f"{metric}_sd"].values[0])                         # SD value
        else:
            y_vals.append(np.nan)                                                 # missing value safeguard
            y_errs.append(0)                                                      # no error bar if missing

    return y_vals, y_errs                                                        # return ordered arrays

def _plot_apa(metric, variants, labels, colors, markers, fig_number,
              title_text, ylabel, filename, note_text, legend_outside=False):    # reusable APA-style plot
    fig, ax = plt.subplots(figsize=(7.5, 5))                                      # match main-pipeline figure size
    x_pos = list(range(len(EPS_ORDER)))                                           # categorical ε positions

    for var, lab in zip(variants, labels):                                        # one line per condition
        y_vals, y_errs = _series_for_variant(combined, var, metric)               # ordered mean/SD arrays

        ax.errorbar(
            x_pos, y_vals, yerr=y_errs,
            marker=markers[var], markersize=6, capsize=3,
            label=lab, color=colors[var], linewidth=1.5,
            markerfacecolor="white", markeredgecolor=colors[var],
            markeredgewidth=1.5
        )

    ax.axhline(y=0, color="black", linestyle="-", alpha=0.15, linewidth=0.6)      # subtle zero reference
    ax.set_xticks(x_pos)                                                         # fixed ε tick positions
    ax.set_xticklabels(EPS_LABELS)                                                # fixed ε tick labels
    ax.set_xlabel("Privacy budget (ε)")                                           # x-axis label
    ax.set_ylabel(ylabel)                                                         # y-axis label

    if legend_outside:                                                            # move legend outside plot area
        leg = ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5),            # place legend to the right
                        borderaxespad=0)
        plt.subplots_adjust(top=0.86, bottom=0.12, right=0.78)                   # make room for outside legend
    else:
        leg = ax.legend(loc="best")                                               # keep legend inside for other plots
        plt.subplots_adjust(top=0.86, bottom=0.12)                                # normal spacing

    leg.get_frame().set_linewidth(0)                                              # no legend border
    leg.get_frame().set_facecolor("none")                                         # transparent legend background

    apa_figure_title(fig, fig_number, title_text, note_text=note_text)            # APA header and note

    path = SWEET_RES / "figures" / filename                                      # output path
    plt.savefig(path)                                                            # save at 300 DPI via rcParams
    plt.show()                                                                   # display
    print(f"Saved: {path}")                                                       # confirmation

# --- Size axis: ρ primary, Qini companion ---
size_v = ["size_25", "size_50", "size_60", "size_70", "size_75", "size_100"]      # size variants incl. infill
size_l = ["25%", "50%", "60%", "70%", "75%", "100%"]                              # legend labels

_plot_apa(
    metric="rho",
    variants=size_v,
    labels=size_l,
    colors=SIZE_COLORS,
    markers=SIZE_MARKERS,
    fig_number="A2",
    title_text="Treatment-Effect Ranking Preservation Across Privacy Levels by Dataset Size",
    ylabel="Spearman ρ (mean ± SD)",
    filename="fig_sweetspot_size_rho.png",
    note_text="Error bars represent ± 1 SD across 10 Monte Carlo replications. The 60% and 70% conditions are exploratory infill variants.",
    legend_outside=True
)

_plot_apa(
    metric="qini",
    variants=size_v,
    labels=size_l,
    colors=SIZE_COLORS,
    markers=SIZE_MARKERS,
    fig_number="A3",
    title_text="Downstream Uplift-Model Performance Across Privacy Levels by Dataset Size",
    ylabel="Qini coefficient (mean ± SD)",
    filename="fig_sweetspot_size_qini.png",
    note_text="Error bars represent ± 1 SD across 10 Monte Carlo replications. Negative Qini values indicate underperformance relative to random targeting.",
    legend_outside=True
)

# --- Imbalance axis: ρ primary, Qini companion ---
imb_v = ["imbalance_05", "imbalance_10", "imbalance_20", "imbalance_35", "imbalance_baseline"]  # imbalance variants incl. infill
imb_l = ["5% newbie", "10% newbie", "20% newbie", "35% newbie", "Baseline (50%)"]               # legend labels

_plot_apa(
    metric="rho",
    variants=imb_v,
    labels=imb_l,
    colors=IMB_COLORS,
    markers=IMB_MARKERS,
    fig_number="A4",
    title_text="Treatment-Effect Ranking Preservation Across Privacy Levels by Subgroup Imbalance",
    ylabel="Spearman ρ (mean ± SD)",
    filename="fig_sweetspot_imbalance_rho.png",
    note_text="Error bars represent ± 1 SD across 10 Monte Carlo replications. The 10% and 35% conditions are exploratory infill variants."
)

_plot_apa(
    metric="qini",
    variants=imb_v,
    labels=imb_l,
    colors=IMB_COLORS,
    markers=IMB_MARKERS,
    fig_number="A5",
    title_text="Downstream Uplift-Model Performance Across Privacy Levels by Subgroup Imbalance",
    ylabel="Qini coefficient (mean ± SD)",
    filename="fig_sweetspot_imbalance_qini.png",
    note_text="Error bars represent ± 1 SD across 10 Monte Carlo replications. Negative Qini values indicate underperformance relative to random targeting."
)


TABLE A16: Full ε-grid results for the size infill variants
Variant    ε     N  Mean ρ   SD ρ  Mean Qini  SD Qini Degenerate runs
size_60  0.1 26880  0.3234 0.1746     -11.74    17.13            8/10
size_60  0.5 26880  0.0936 0.2949      -2.14    27.89            0/10
size_60  1.0 26880  0.2410 0.1781      20.00    18.13            0/10
size_60  2.0 26880  0.2756 0.0237      30.68     8.63            0/10
size_60  5.0 26880  0.2669 0.0138      33.23     5.25            0/10
size_60 10.0 26880  0.2682 0.0105      34.85     4.14            0/10
size_60    ∞ 26880  0.2639 0.0102      33.25     4.99            0/10
size_70  0.1 31359  0.2103 0.2309     -10.22    22.63            6/10
size_70  0.5 31359  0.3086 0.0493      19.68    11.76            0/10
size_70  1.0 31359  0.3257 0.0489      21.96    10.58            0/10
size_70  2.0 31359  0.2867 0.0396      28.16     7.51            0/10
size_70  5.0 31359  0.2750 0.0159      31.55     4.23            0/10
size_70 10.0 31359  0.2710 0.